# Collaborative Filtering: A Comprehensive Guide

---

## 1. Introduction

**Collaborative Filtering (CF)** is one of the most successful and widely deployed techniques in recommender systems. The fundamental idea is elegantly simple: *predict a user's interests by collecting preference information from many users (collaborating)*.

Unlike content-based filtering, which relies on item attributes (genre, color, brand), collaborative filtering leverages the **collective wisdom of users** — the assumption being that if User A and User B agreed in the past, they are likely to agree again in the future.

### 1.1 Historical Context

The term "collaborative filtering" was coined by David Goldberg et al. in 1992 in their seminal paper on **Tapestry**, an electronic messaging system at Xerox PARC. The field gained mainstream attention with:

- **GroupLens (1994)** — Usenet news article recommendations
- **Amazon (1998)** — "Customers who bought this also bought..."
- **Netflix Prize (2006–2009)** — $1M competition that revolutionized the field

### 1.2 The Core Assumption

Collaborative filtering rests on the **homophily hypothesis**:

$$\text{If } \text{sim}(u_i, u_j) \text{ is high, then } r_{u_i, k} \approx r_{u_j, k}$$

where $$\text{sim}(u_i, u_j)$$ denotes the similarity between users $$u_i$$ and $$u_j$$, and $$r_{u,k}$$ is the rating of user $$u$$ for item $$k$$.

### 1.3 The User-Item Interaction Matrix

At the heart of CF lies the **user-item interaction matrix** $$R \in \mathbb{R}^{m \times n}$$, where:
- $$m$$ = number of users
- $$n$$ = number of items
- $$r_{ij}$$ = rating/interaction of user $$i$$ for item $$j$$

This matrix is typically **extremely sparse** (Netflix: ~1% filled, Amazon: ~0.01% filled). The goal of CF is to **fill in the missing entries** of this matrix.

### 1.4 Types of Feedback

| Type | Description | Examples |
|------|-------------|----------|
| Explicit | Direct user ratings | 1-5 stars, thumbs up/down |
| Implicit | Inferred from behavior | Clicks, purchases, watch time, page views |
| Unary | Binary signal | Purchased/not, clicked/not |

## 2. Taxonomy of Collaborative Filtering Approaches

Collaborative filtering methods are broadly categorized into two families:

```
                    Collaborative Filtering
                            │
              ┌────────────┴────────────┐
              │                         │
       Memory-Based                Model-Based
       (Neighborhood)              (Latent Factor)
              │                         │
      ┌──────┴──────┐         ┌──────┴─────────────┐
      │             │         │         │             │
  User-Based   Item-Based   SVD/SVD++  ALS    Neural CF
      CF           CF        NMF     BPR    Autoencoders
```

### 2.1 Memory-Based (Neighborhood) Methods

**Philosophy:** Use the entire user-item matrix directly. Find similar users/items and aggregate their ratings.

| Aspect | Description |
|--------|-------------|
| Approach | Heuristic, similarity-based |
| Scalability | Poor for large datasets ($$O(m^2)$$ or $$O(n^2)$$) |
| Interpretability | High — "recommended because users like you also liked..." |
| Cold Start | Struggles with new users/items |
| Industry Use | Amazon (early), GroupLens |

### 2.2 Model-Based (Latent Factor) Methods

**Philosophy:** Learn a compressed representation (latent factors) from the data. Map users and items to a shared latent space.

| Aspect | Description |
|--------|-------------|
| Approach | Optimization/learning-based |
| Scalability | Good — models are compact once trained |
| Interpretability | Lower — latent dimensions are abstract |
| Cold Start | Can incorporate side features |
| Industry Use | Netflix, Spotify, YouTube |

## 3. User-Based Collaborative Filtering

### 3.1 Intuition

User-Based CF answers: *"Users who are similar to you liked item X, so you might like it too."*

**Industrial Example — Amazon (early 2000s):** When you browse a product, Amazon identifies users with similar purchase/rating histories and recommends items those similar users bought but you haven't.

### 3.2 Algorithm Steps

1. Compute pairwise similarity between the target user and all other users
2. Select the top-$$k$$ most similar users (neighborhood)
3. Predict the target user's rating for unrated items using a weighted average of neighbors' ratings

### 3.3 Similarity Metrics

**Pearson Correlation Coefficient:**

$$\text{sim}(u, v) = \frac{\sum_{i \in I_{uv}} (r_{ui} - \bar{r}_u)(r_{vi} - \bar{r}_v)}{\sqrt{\sum_{i \in I_{uv}} (r_{ui} - \bar{r}_u)^2} \cdot \sqrt{\sum_{i \in I_{uv}} (r_{vi} - \bar{r}_v)^2}}$$

where $$I_{uv}$$ is the set of items rated by both users $$u$$ and $$v$$, and $$\bar{r}_u$$ is user $$u$$'s mean rating.

**Cosine Similarity:**

$$\text{sim}(u, v) = \frac{\mathbf{r}_u \cdot \mathbf{r}_v}{\|\mathbf{r}_u\| \cdot \|\mathbf{r}_v\|} = \frac{\sum_{i \in I_{uv}} r_{ui} \cdot r_{vi}}{\sqrt{\sum_{i \in I_{uv}} r_{ui}^2} \cdot \sqrt{\sum_{i \in I_{uv}} r_{vi}^2}}$$

**Adjusted Cosine Similarity** (accounts for rating scale differences):

$$\text{sim}(u, v) = \frac{\sum_{i \in I_{uv}} (r_{ui} - \bar{r}_i)(r_{vi} - \bar{r}_i)}{\sqrt{\sum_{i \in I_{uv}} (r_{ui} - \bar{r}_i)^2} \cdot \sqrt{\sum_{i \in I_{uv}} (r_{vi} - \bar{r}_i)^2}}$$

### 3.4 Rating Prediction

The predicted rating of user $$u$$ for item $$i$$:

$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in N_k(u)} \text{sim}(u, v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in N_k(u)} |\text{sim}(u, v)|}$$

where $$N_k(u)$$ is the set of $$k$$ nearest neighbors of user $$u$$ who have rated item $$i$$.

### 3.5 Complexity Analysis

- **Time:** $$O(m^2 \cdot n)$$ for computing all pairwise user similarities
- **Space:** $$O(m^2)$$ for storing the similarity matrix
- **Prediction:** $$O(k)$$ per prediction (with precomputed similarities)

### 3.6 Industrial Example: GroupLens / MovieLens

The GroupLens research group at the University of Minnesota pioneered User-Based CF for Usenet news recommendations. Their MovieLens dataset (100K–25M ratings) remains the benchmark for CF research.

In [0]:
%pip install scikit-surprise --quiet
dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr

# =============================================================================
# Create a sample User-Item Rating Matrix (Movie Ratings Scenario)
# Simulates a small Netflix/MovieLens-like dataset
# =============================================================================

# Users represent Netflix subscribers; Items represent movies
ratings_data = {
    'User':  ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace'],
    'Toy Story': [5, 4, 1, 2, 5, 4, np.nan],
    'Titanic':   [3, np.nan, 4, 5, 2, np.nan, 4],
    'The Matrix':[4, 5, 2, 1, 4, 5, np.nan],
    'Inception': [np.nan, 3, 5, 4, np.nan, 2, 5],
    'Frozen':    [5, np.nan, 2, 1, 4, np.nan, 3],
    'Interstellar': [np.nan, 4, 3, np.nan, 5, 4, 2],
    'The Godfather': [2, 5, np.nan, 3, 1, 5, 4],
}

df_ratings = pd.DataFrame(ratings_data).set_index('User')
print("=" * 70)
print("USER-ITEM RATING MATRIX (NaN = unrated)")
print("=" * 70)
display(df_ratings)
print(f"\nMatrix dimensions: {df_ratings.shape[0]} users × {df_ratings.shape[1]} items")
print(f"Sparsity: {df_ratings.isna().sum().sum() / (df_ratings.shape[0] * df_ratings.shape[1]):.1%}")

In [0]:
# =============================================================================
# USER-BASED COLLABORATIVE FILTERING - Full Implementation
# =============================================================================

class UserBasedCF:
    """
    User-Based Collaborative Filtering with Pearson Correlation.
    
    This implements the classic kNN approach used in early Amazon and 
    GroupLens recommender systems.
    """
    
    def __init__(self, ratings_matrix, k=3):
        """
        Args:
            ratings_matrix: DataFrame with users as rows, items as columns
            k: Number of nearest neighbors to consider
        """
        self.ratings = ratings_matrix
        self.k = k
        self.user_means = ratings_matrix.mean(axis=1)  # Mean rating per user
        self.similarity_matrix = self._compute_similarity_matrix()
    
    def _pearson_similarity(self, user_a, user_b):
        """Compute Pearson correlation between two users over co-rated items."""
        # Find items rated by both users
        mask = self.ratings.loc[user_a].notna() & self.ratings.loc[user_b].notna()
        co_rated = mask.sum()
        
        if co_rated < 2:  # Need at least 2 co-rated items
            return 0.0
        
        ratings_a = self.ratings.loc[user_a][mask]
        ratings_b = self.ratings.loc[user_b][mask]
        
        # Pearson correlation
        mean_a = ratings_a.mean()
        mean_b = ratings_b.mean()
        
        numerator = ((ratings_a - mean_a) * (ratings_b - mean_b)).sum()
        denom_a = np.sqrt(((ratings_a - mean_a) ** 2).sum())
        denom_b = np.sqrt(((ratings_b - mean_b) ** 2).sum())
        
        if denom_a == 0 or denom_b == 0:
            return 0.0
        
        return numerator / (denom_a * denom_b)
    
    def _compute_similarity_matrix(self):
        """Compute pairwise user similarity matrix."""
        users = self.ratings.index
        n_users = len(users)
        sim_matrix = pd.DataFrame(np.zeros((n_users, n_users)), 
                                  index=users, columns=users)
        
        for i, user_a in enumerate(users):
            for j, user_b in enumerate(users):
                if i < j:
                    sim = self._pearson_similarity(user_a, user_b)
                    sim_matrix.loc[user_a, user_b] = sim
                    sim_matrix.loc[user_b, user_a] = sim
                elif i == j:
                    sim_matrix.loc[user_a, user_b] = 1.0
        
        return sim_matrix
    
    def predict(self, user, item):
        """Predict rating for a user-item pair."""
        if pd.notna(self.ratings.loc[user, item]):
            return self.ratings.loc[user, item]  # Already rated
        
        # Find users who rated this item
        raters = self.ratings[item].dropna().index
        raters = [u for u in raters if u != user]
        
        if not raters:
            return self.user_means[user]  # Fallback to user mean
        
        # Get similarities and select top-k neighbors
        similarities = [(other, self.similarity_matrix.loc[user, other]) 
                       for other in raters]
        similarities.sort(key=lambda x: abs(x[1]), reverse=True)
        neighbors = similarities[:self.k]
        
        # Weighted average prediction
        numerator = sum(sim * (self.ratings.loc[other, item] - self.user_means[other])
                       for other, sim in neighbors)
        denominator = sum(abs(sim) for _, sim in neighbors)
        
        if denominator == 0:
            return self.user_means[user]
        
        prediction = self.user_means[user] + numerator / denominator
        return np.clip(prediction, 1, 5)  # Clip to valid rating range
    
    def recommend(self, user, n=3):
        """Generate top-n recommendations for a user."""
        unrated_items = self.ratings.loc[user][self.ratings.loc[user].isna()].index
        predictions = [(item, self.predict(user, item)) for item in unrated_items]
        predictions.sort(key=lambda x: x[1], reverse=True)
        return predictions[:n]


# --- Run the User-Based CF ---
model = UserBasedCF(df_ratings, k=3)

print("=" * 70)
print("USER SIMILARITY MATRIX (Pearson Correlation)")
print("=" * 70)
display(model.similarity_matrix.round(3))

print("\n" + "=" * 70)
print("PREDICTIONS & RECOMMENDATIONS")
print("=" * 70)

# Predict Grace's rating for Toy Story
pred = model.predict('Grace', 'Toy Story')
print(f"\nPredicted rating for Grace → 'Toy Story': {pred:.2f}")

# Get top recommendations for Bob
recs = model.recommend('Bob', n=3)
print(f"\nTop-3 Recommendations for Bob:")
for item, score in recs:
    print(f"  • {item}: predicted rating = {score:.2f}")

## 4. Item-Based Collaborative Filtering

### 4.1 Intuition

Item-Based CF answers: *"You liked item A, and item B is similar to item A, so you might like item B too."*

The key insight (from Amazon's 2003 paper by Linden, Smith & York): **item-item similarities are more stable than user-user similarities** because items don't change, but user preferences evolve over time.

### 4.2 Why Item-Based Over User-Based?

| Criterion | User-Based | Item-Based |
|-----------|-----------|------------|
| Stability | User tastes change | Item properties are static |
| Scalability | $$O(m^2)$$ users | $$O(n^2)$$ items (often $$n \ll m$$) |
| Precomputation | Must recompute frequently | Can precompute offline |
| Interpretability | "Users like you..." | "Because you liked X..." |

### 4.3 Algorithm

**Step 1: Compute Item-Item Similarity (Adjusted Cosine)**

$$\text{sim}(i, j) = \frac{\sum_{u \in U_{ij}} (r_{ui} - \bar{r}_u)(r_{uj} - \bar{r}_u)}{\sqrt{\sum_{u \in U_{ij}} (r_{ui} - \bar{r}_u)^2} \cdot \sqrt{\sum_{u \in U_{ij}} (r_{uj} - \bar{r}_u)^2}}$$

where $$U_{ij}$$ is the set of users who rated both items $$i$$ and $$j$$.

**Step 2: Predict Rating**

$$\hat{r}_{ui} = \frac{\sum_{j \in N_k(i)} \text{sim}(i, j) \cdot r_{uj}}{\sum_{j \in N_k(i)} |\text{sim}(i, j)|}$$

where $$N_k(i)$$ is the set of $$k$$ items most similar to item $$i$$ that user $$u$$ has rated.

### 4.4 Industrial Example: Amazon.com

**"Customers who bought this item also bought..."**

Amazon's item-to-item collaborative filtering (patented 2001) is one of the most commercially successful recommendation algorithms:

- Precomputes an item-item similarity table offline
- At recommendation time, looks up items similar to the user's recent purchases
- Scales to hundreds of millions of items and users
- Responsible for an estimated **35% of Amazon's revenue**

The algorithm's genius is its **sub-linear online complexity**: once the similarity table is built, generating recommendations is $$O(k)$$ per item in the user's history.

In [0]:
# =============================================================================
# ITEM-BASED COLLABORATIVE FILTERING
# Implements the approach used by Amazon.com
# =============================================================================

class ItemBasedCF:
    """
    Item-Based Collaborative Filtering with Adjusted Cosine Similarity.
    
    This is the approach described in Amazon's seminal 2003 paper:
    'Amazon.com Recommendations: Item-to-Item Collaborative Filtering'
    """
    
    def __init__(self, ratings_matrix, k=3):
        self.ratings = ratings_matrix
        self.k = k
        self.user_means = ratings_matrix.mean(axis=1)
        self.item_similarity = self._compute_item_similarity()
    
    def _adjusted_cosine_similarity(self, item_a, item_b):
        """Adjusted cosine similarity between two items."""
        # Find users who rated both items
        mask = self.ratings[item_a].notna() & self.ratings[item_b].notna()
        co_raters = mask[mask].index
        
        if len(co_raters) < 2:
            return 0.0
        
        # Mean-center ratings by user
        centered_a = self.ratings.loc[co_raters, item_a] - self.user_means[co_raters]
        centered_b = self.ratings.loc[co_raters, item_b] - self.user_means[co_raters]
        
        numerator = (centered_a * centered_b).sum()
        denom = np.sqrt((centered_a ** 2).sum()) * np.sqrt((centered_b ** 2).sum())
        
        if denom == 0:
            return 0.0
        return numerator / denom
    
    def _compute_item_similarity(self):
        """Compute pairwise item similarity matrix."""
        items = self.ratings.columns
        n_items = len(items)
        sim_matrix = pd.DataFrame(np.zeros((n_items, n_items)),
                                  index=items, columns=items)
        
        for i in range(n_items):
            for j in range(i + 1, n_items):
                sim = self._adjusted_cosine_similarity(items[i], items[j])
                sim_matrix.iloc[i, j] = sim
                sim_matrix.iloc[j, i] = sim
            sim_matrix.iloc[i, i] = 1.0
        
        return sim_matrix
    
    def predict(self, user, item):
        """Predict rating using weighted sum of similar items the user rated."""
        if pd.notna(self.ratings.loc[user, item]):
            return self.ratings.loc[user, item]
        
        # Items the user has rated
        rated_items = self.ratings.loc[user].dropna().index
        rated_items = [it for it in rated_items if it != item]
        
        if not rated_items:
            return self.user_means[user]
        
        # Get similarities to the target item
        similarities = [(it, self.item_similarity.loc[item, it]) for it in rated_items]
        similarities.sort(key=lambda x: abs(x[1]), reverse=True)
        neighbors = similarities[:self.k]
        
        numerator = sum(sim * self.ratings.loc[user, it] for it, sim in neighbors)
        denominator = sum(abs(sim) for _, sim in neighbors)
        
        if denominator == 0:
            return self.user_means[user]
        
        return np.clip(numerator / denominator, 1, 5)
    
    def recommend(self, user, n=3):
        """Generate top-n item recommendations."""
        unrated = self.ratings.loc[user][self.ratings.loc[user].isna()].index
        preds = [(item, self.predict(user, item)) for item in unrated]
        preds.sort(key=lambda x: x[1], reverse=True)
        return preds[:n]


# --- Run Item-Based CF ---
item_model = ItemBasedCF(df_ratings, k=3)

print("=" * 70)
print("ITEM-ITEM SIMILARITY MATRIX (Adjusted Cosine)")
print("=" * 70)
display(item_model.item_similarity.round(3))

print("\n" + "=" * 70)
print("ITEM-BASED CF RECOMMENDATIONS")
print("=" * 70)

# Recommendations for Bob
recs_bob = item_model.recommend('Bob', n=3)
print(f"\nTop-3 Recommendations for Bob (Item-Based):")
for item, score in recs_bob:
    print(f"  • {item}: predicted rating = {score:.2f}")

# Compare with User-Based
recs_bob_user = model.recommend('Bob', n=3)
print(f"\nTop-3 Recommendations for Bob (User-Based):")
for item, score in recs_bob_user:
    print(f"  • {item}: predicted rating = {score:.2f}")

print("\n💡 Note: Item-Based and User-Based may give different recommendations!")
print("   Item-Based focuses on item relationships; User-Based on user similarity.")

## 5. Matrix Factorization & SVD

### 5.1 The Dimensionality Problem

Memory-based methods suffer from:
1. **Sparsity** — too few co-rated items to compute meaningful similarity
2. **Scalability** — quadratic complexity in users or items
3. **Noise** — all ratings used equally, even noisy ones

Matrix Factorization (MF) solves these by discovering **latent factors** that explain observed ratings.

### 5.2 Classic SVD Decomposition

For the fully observed rating matrix $$R \in \mathbb{R}^{m \times n}$$, the **Singular Value Decomposition** is:

$$R = U \Sigma V^T$$

where:
- $$U \in \mathbb{R}^{m \times r}$$ — left singular vectors (user factors)
- $$\Sigma \in \mathbb{R}^{r \times r}$$ — diagonal matrix of singular values
- $$V \in \mathbb{R}^{n \times r}$$ — right singular vectors (item factors)
- $$r$$ — rank of the matrix

**Truncated SVD (rank-$$k$$ approximation):**

$$\hat{R}_k = U_k \Sigma_k V_k^T$$

The Eckart-Young theorem guarantees this is the **best rank-$$k$$ approximation** in terms of Frobenius norm:

$$\hat{R}_k = \arg\min_{\text{rank}(M) \leq k} \|R - M\|_F$$

### 5.3 The Funk SVD (Simon Funk, 2006)

During the Netflix Prize, Simon Funk popularized a practical variant that handles **missing entries** by learning factors through optimization:

**Objective:** Minimize the regularized squared error over observed ratings:

$$\min_{P, Q} \sum_{(u,i) \in \mathcal{K}} \left(r_{ui} - p_u^T q_i\right)^2 + \lambda \left(\|p_u\|^2 + \|q_i\|^2\right)$$

where:
- $$p_u \in \mathbb{R}^k$$ — user $$u$$'s latent factor vector
- $$q_i \in \mathbb{R}^k$$ — item $$i$$'s latent factor vector
- $$\lambda$$ — regularization coefficient
- $$\mathcal{K}$$ — set of observed (user, item) pairs

**Predicted rating:**

$$\hat{r}_{ui} = p_u^T q_i = \sum_{f=1}^{k} p_{uf} \cdot q_{if}$$

### 5.4 SVD++ Extension (Koren, 2008)

SVD++ incorporates **implicit feedback** (items a user has rated, regardless of the rating value):

$$\hat{r}_{ui} = \mu + b_u + b_i + \left(p_u + |I_u|^{-1/2} \sum_{j \in I_u} y_j\right)^T q_i$$

where:
- $$\mu$$ — global average rating
- $$b_u$$ — user bias (e.g., harsh vs. lenient rater)
- $$b_i$$ — item bias (e.g., popular vs. niche items)
- $$I_u$$ — set of items rated by user $$u$$
- $$y_j$$ — implicit feedback factor for item $$j$$

### 5.5 SGD Update Rules

For Funk SVD with learning rate $$\gamma$$, define the error:

$$e_{ui} = r_{ui} - \hat{r}_{ui}$$

Update rules (gradient descent):

$$p_u \leftarrow p_u + \gamma \left(e_{ui} \cdot q_i - \lambda \cdot p_u\right)$$

$$q_i \leftarrow q_i + \gamma \left(e_{ui} \cdot p_u - \lambda \cdot q_i\right)$$

### 5.6 Industrial Example: Netflix Prize Winner

The Netflix Prize (2006–2009) showed that matrix factorization dramatically outperforms neighborhood methods. The winning solution "BellKor's Pragmatic Chaos" combined 107 different models, with SVD++ at its core. Netflix reduced RMSE from 0.9514 (Cinematch baseline) to 0.8567 — a 10.06% improvement.

In [0]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# =============================================================================
# PART 1: Truncated SVD for Visualization
# =============================================================================

# Create a denser synthetic dataset for meaningful SVD
np.random.seed(42)
n_users, n_items = 50, 20

# Simulate 3 latent factors: action lovers, romance lovers, sci-fi lovers
user_prefs = np.random.randn(n_users, 3)      # User taste profiles
item_factors = np.random.randn(n_items, 3)    # Item genre profiles
true_ratings = user_prefs @ item_factors.T
true_ratings = 1 + 4 * (true_ratings - true_ratings.min()) / (true_ratings.max() - true_ratings.min())

# Introduce sparsity (observe only 40% of entries)
mask = np.random.rand(n_users, n_items) < 0.4
observed = np.where(mask, true_ratings, 0)

# Column/row names
user_ids = [f'U{i+1:02d}' for i in range(n_users)]
item_ids = [f'Item_{i+1:02d}' for i in range(n_items)]
R = pd.DataFrame(observed, index=user_ids, columns=item_ids)

# --- Truncated SVD ---
svd = TruncatedSVD(n_components=3, random_state=42)
U = svd.fit_transform(R.values)           # User latent factors
S = np.diag(svd.singular_values_)        # Singular values
Vt = svd.components_                      # Item latent factors
R_approx = U @ Vt                        # Low-rank approximation

print("=" * 70)
print("SVD DECOMPOSITION RESULTS")
print("=" * 70)
print(f"\nOriginal matrix shape: {R.shape}")
print(f"U shape (user factors): {U.shape}")
print(f"S (singular values):    {svd.singular_values_.round(2)}")
print(f"Vt shape (item factors): {Vt.shape}")
print(f"\nVariance explained by 3 components: {svd.explained_variance_ratio_.sum():.1%}")

# Reconstruction error
frob_error = np.linalg.norm(R.values - R_approx, 'fro')
print(f"Frobenius reconstruction error (3 components): {frob_error:.4f}")

# --- Plot singular value decay ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

svd_full = TruncatedSVD(n_components=10, random_state=42)
svd_full.fit(R.values)
axes[0].bar(range(1, 11), svd_full.singular_values_, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Component')
axes[0].set_ylabel('Singular Value')
axes[0].set_title('Singular Value Decay\n(Information Content per Component)')
axes[0].set_xticks(range(1, 11))

# Variance explained
cumvar = np.cumsum(svd_full.explained_variance_ratio_)
axes[1].plot(range(1, 11), cumvar * 100, 'o-', color='darkorange', linewidth=2)
axes[1].axhline(y=80, color='red', linestyle='--', alpha=0.5, label='80% threshold')
axes[1].set_xlabel('Number of Components (k)')
axes[1].set_ylabel('Cumulative Variance Explained (%)')
axes[1].set_title('Choosing k: Cumulative Variance Explained')
axes[1].legend()
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
display(fig)
plt.close(fig)

In [0]:
# =============================================================================
# PART 2: Funk SVD with Bias Terms (Simon Funk's Netflix Prize approach)
# =============================================================================

class FunkSVD:
    """
    Funk SVD (2006) with bias terms.
    Implements the core algorithm from the Netflix Prize.
    This is the foundation of modern recommender systems.
    """
    
    def __init__(self, n_factors=20, n_epochs=100, lr=0.005, reg=0.02):
        """
        Args:
            n_factors: Number of latent factors k
            n_epochs:  Number of SGD passes
            lr:        Learning rate (gamma)
            reg:       L2 regularization coefficient (lambda)
        """
        self.k = n_factors
        self.n_epochs = n_epochs
        self.lr = lr
        self.reg = reg
        self.train_losses = []
    
    def fit(self, df_train, n_users, n_items):
        """Train on (user_idx, item_idx, rating) triples."""
        np.random.seed(42)
        # Initialize factors with small random values
        self.P = np.random.normal(0, 0.1, (n_users, self.k))   # User factors
        self.Q = np.random.normal(0, 0.1, (n_items, self.k))   # Item factors
        self.b_u = np.zeros(n_users)                             # User biases
        self.b_i = np.zeros(n_items)                             # Item biases
        self.mu = df_train['rating'].mean()                      # Global mean
        
        for epoch in range(self.n_epochs):
            total_loss = 0
            df_shuffled = df_train.sample(frac=1, random_state=epoch)
            
            for _, row in df_shuffled.iterrows():
                u, i, r = int(row['user']), int(row['item']), row['rating']
                
                # Prediction: mu + b_u + b_i + p_u^T q_i
                pred = self.mu + self.b_u[u] + self.b_i[i] + self.P[u] @ self.Q[i]
                e = r - pred
                total_loss += e ** 2
                
                # Update biases
                self.b_u[u] += self.lr * (e - self.reg * self.b_u[u])
                self.b_i[i] += self.lr * (e - self.reg * self.b_i[i])
                
                # Update latent factors (SGD step)
                p_u_old = self.P[u].copy()
                self.P[u] += self.lr * (e * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (e * p_u_old - self.reg * self.Q[i])
            
            rmse = np.sqrt(total_loss / len(df_train))
            self.train_losses.append(rmse)
        return self
    
    def predict(self, u, i):
        """Predict rating for user u, item i."""
        pred = self.mu + self.b_u[u] + self.b_i[i] + self.P[u] @ self.Q[i]
        return np.clip(pred, 1, 5)
    
    def evaluate_rmse(self, df_test):
        """Compute RMSE on test set."""
        errors = [(r - self.predict(int(u), int(i))) ** 2
                  for u, i, r in zip(df_test['user'], df_test['item'], df_test['rating'])]
        return np.sqrt(np.mean(errors))


# --- Generate a structured synthetic dataset ---
np.random.seed(42)
n_users, n_items = 200, 50

# 5 user groups, 5 item categories (true structure)
user_groups = np.random.randint(0, 5, n_users)
item_categories = np.random.randint(0, 5, n_items)

# Users prefer items in their group (high affinity = 4-5 stars)
records = []
for u in range(n_users):
    for i in range(n_items):
        if np.random.rand() < 0.15:  # ~15% observed
            affinity = 4.5 if user_groups[u] == item_categories[i] else 1.5
            rating = np.clip(affinity + np.random.randn() * 0.5, 1, 5)
            records.append({'user': u, 'item': i, 'rating': round(rating)})

df = pd.DataFrame(records)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Dataset: {n_users} users, {n_items} items, {len(df)} ratings")
print(f"Sparsity: {1 - len(df)/(n_users*n_items):.1%}")

# --- Train Funk SVD ---
funk_svd = FunkSVD(n_factors=10, n_epochs=50, lr=0.005, reg=0.02)
funk_svd.fit(train_df, n_users, n_items)

train_rmse = funk_svd.train_losses[-1]
test_rmse = funk_svd.evaluate_rmse(test_df)

print(f"\nFunk SVD Results (k=10 factors, 50 epochs):")
print(f"  Train RMSE: {train_rmse:.4f}")
print(f"  Test RMSE:  {test_rmse:.4f}")

# Plot training convergence
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(funk_svd.train_losses, color='steelblue', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('RMSE')
ax.set_title('Funk SVD Training Convergence (Netflix Prize Approach)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
display(fig)
plt.close(fig)

print(f"\nSample predictions vs actuals:")
sample = test_df.sample(8, random_state=1)
results = pd.DataFrame({
    'User': sample['user'].values,
    'Item': sample['item'].values,
    'Actual': sample['rating'].values,
    'Predicted': [round(funk_svd.predict(int(u), int(i)), 2)
                  for u, i in zip(sample['user'], sample['item'])]
})
display(results)

## 6. Alternating Least Squares (ALS)

### 6.1 Motivation: From SGD to ALS

Funk SVD with SGD is sequential and hard to parallelize. **ALS** reformulates the same optimization problem in a way that's **highly parallelizable** — making it the algorithm of choice for distributed systems like Apache Spark.

### 6.2 The ALS Objective

Minimize the same Frobenius-norm objective:

$$\mathcal{L}(P, Q) = \sum_{(u,i) \in \mathcal{K}} \left(r_{ui} - p_u^T q_i\right)^2 + \lambda \left(\sum_u \|p_u\|^2 + \sum_i \|q_i\|^2\right)$$

### 6.3 Alternating Optimization

The objective is **non-convex jointly** in $$(P, Q)$$, but **convex in each one when the other is fixed**. ALS exploits this:

**Step 1 — Fix $$Q$$, solve for each $$p_u$$:**

$$p_u = \left( Q_{I_u}^T Q_{I_u} + \lambda |I_u| \mathbf{I} \right)^{-1} Q_{I_u}^T r_u$$

where $$Q_{I_u}$$ is the submatrix of $$Q$$ for items rated by user $$u$$, and $$r_u$$ is the vector of user $$u$$'s known ratings.

**Step 2 — Fix $$P$$, solve for each $$q_i$$:**

$$q_i = \left( P_{U_i}^T P_{U_i} + \lambda |U_i| \mathbf{I} \right)^{-1} P_{U_i}^T r_i$$

where $$U_i$$ is the set of users who rated item $$i$$.

Each step is a **closed-form least squares solution** and each user/item update is **independent** — hence massively parallelizable.

### 6.4 Implicit ALS (Hu, Koren, Volinsky, 2008)

For implicit feedback (e.g., play counts, page views), binary confidence values are used:

$$c_{ui} = 1 + \alpha \cdot d_{ui}$$

where $$d_{ui}$$ is the raw observation count, and preference:

$$p_{ui} = \begin{cases} 1 & \text{if } d_{ui} > 0 \\ 0 & \text{otherwise} \end{cases}$$

Objective:

$$\mathcal{L}(P, Q) = \sum_{u,i} c_{ui}\left(p_{ui} - p_u^T q_i\right)^2 + \lambda \left(\sum_u \|p_u\|^2 + \sum_i \|q_i\|^2\right)$$

This is the algorithm powering **Spotify's Discover Weekly**.

### 6.5 Weighted ALS Update (Implicit)

$$p_u = \left( Q^T C^u Q + \lambda \mathbf{I} \right)^{-1} Q^T C^u \mathbf{p}_u$$

where $$C^u = \text{diag}(c_{u1}, c_{u2}, \ldots, c_{un})$$ is the confidence diagonal matrix for user $$u$$.

The trick to make this $$O(k^2 n + k^3)$$ instead of $$O(n^2 k)$$ is precomputing $$Q^T Q$$:

$$Q^T C^u Q = Q^T Q + Q^T (C^u - I) Q$$

### 6.6 Industrial Example: Spotify Discover Weekly

Spotify uses Implicit ALS on 40M+ users and 30M+ tracks, trained on streaming play counts:
- $$\alpha = 40$$ (confidence scaling factor)
- $$k = 50$$ latent factors
- Trained weekly on a Hadoop/Spark cluster
- Generates 30-song personalized playlists delivered every Monday
- Powers ~30% of Spotify's total listening hours

In [0]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType

# =============================================================================
# ALS COLLABORATIVE FILTERING WITH APACHE SPARK MLlib
# This is the production-grade approach used at Spotify, Netflix, and LinkedIn
# =============================================================================

# --- Create a synthetic ratings DataFrame in Spark ---
np.random.seed(42)
n_users, n_items = 500, 100
user_groups = np.random.randint(0, 5, n_users)
item_categories = np.random.randint(0, 5, n_items)

spark_records = []
for u in range(n_users):
    for i in range(n_items):
        if np.random.rand() < 0.08:  # ~8% observed (streaming scenario)
            affinity = 4.5 if user_groups[u] == item_categories[i] else 1.5
            rating = float(np.clip(affinity + np.random.randn() * 0.7, 1, 5))
            spark_records.append((u, i, rating))

schema = StructType([
    StructField('userId', IntegerType(), False),
    StructField('itemId', IntegerType(), False),
    StructField('rating', FloatType(), False),
])
ratings_df = spark.createDataFrame(spark_records, schema=schema)

print("=" * 70)
print("SPARK ALS - DATASET SUMMARY")
print("=" * 70)
ratings_df.groupBy().agg(
    F.count('rating').alias('total_ratings'),
    F.countDistinct('userId').alias('n_users'),
    F.countDistinct('itemId').alias('n_items'),
    F.round(F.avg('rating'), 3).alias('avg_rating'),
    F.round(F.min('rating'), 2).alias('min_rating'),
    F.round(F.max('rating'), 2).alias('max_rating')
).show()

# --- Train/Test Split ---
train_spark, test_spark = ratings_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train size: {train_spark.count():,} | Test size: {test_spark.count():,}")

# --- Configure and Train ALS ---
als = ALS(
    maxIter=15,           # Number of ALS iterations
    rank=10,              # Number of latent factors (k)
    regParam=0.05,        # Regularization lambda
    userCol='userId',
    itemCol='itemId',
    ratingCol='rating',
    coldStartStrategy='drop',   # Handle unseen users/items in test set
    nonnegative=False,
    seed=42
)

print("\nTraining ALS model...")
model = als.fit(train_spark)
print("Training complete!")

# --- Evaluate ---
predictions = model.transform(test_spark)
evaluator_rmse = RegressionEvaluator(
    metricName='rmse', labelCol='rating', predictionCol='prediction'
)
evaluator_mae = RegressionEvaluator(
    metricName='mae', labelCol='rating', predictionCol='prediction'
)

rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print("\n" + "=" * 70)
print("ALS EVALUATION RESULTS")
print("=" * 70)
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")

# --- Generate Top-K Recommendations ---
print("\n" + "=" * 70)
print("TOP-5 RECOMMENDATIONS FOR FIRST 5 USERS")
print("=" * 70)
userRecs = model.recommendForAllUsers(5)
userRecs.select('userId', 'recommendations').limit(5).show(truncate=False)

print("\n" + "=" * 70)
print("TOP-5 USERS FOR FIRST 5 ITEMS (Who should see each item?)")
print("=" * 70)
itemRecs = model.recommendForAllItems(5)
itemRecs.select('itemId', 'recommendations').limit(5).show(truncate=False)

In [0]:
# =============================================================================
# ALS HYPERPARAMETER ANALYSIS
# Demonstrates impact of rank (k) and regularization (lambda)
# =============================================================================

results = []
ranks = [5, 10, 20, 50]
reg_params = [0.01, 0.05, 0.1]

for rank in ranks:
    for reg in reg_params:
        als_exp = ALS(
            maxIter=10, rank=rank, regParam=reg,
            userCol='userId', itemCol='itemId', ratingCol='rating',
            coldStartStrategy='drop', seed=42
        )
        m = als_exp.fit(train_spark)
        preds = m.transform(test_spark)
        rmse_val = evaluator_rmse.evaluate(preds)
        results.append({'rank': rank, 'reg': reg, 'rmse': round(rmse_val, 4)})

results_df = pd.DataFrame(results)
pivot = results_df.pivot(index='rank', columns='reg', values='rmse')

print("=" * 70)
print("ALS HYPERPARAMETER GRID: RMSE on Test Set")
print("=" * 70)
print("\nColumns = regularization (lambda), Rows = rank (k)\n")
print(pivot.to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(range(len(reg_params)))
ax.set_xticklabels([f"λ={r}" for r in reg_params])
ax.set_yticks(range(len(ranks)))
ax.set_yticklabels([f"k={r}" for r in ranks])
plt.colorbar(im, ax=ax, label='RMSE')
for i in range(len(ranks)):
    for j in range(len(reg_params)):
        ax.text(j, i, f"{pivot.values[i,j]:.3f}", ha='center', va='center',
                fontsize=10, color='black')
ax.set_title('ALS Hyperparameter Grid Search\n(lower RMSE = better)')
ax.set_xlabel('Regularization (λ)')
ax.set_ylabel('Rank (k)')
plt.tight_layout()
display(fig)
plt.close(fig)

best = results_df.loc[results_df['rmse'].idxmin()]
print(f"\nBest configuration: rank={int(best['rank'])}, lambda={best['reg']}, RMSE={best['rmse']}")

## 7. Non-Negative Matrix Factorization (NMF)

### 7.1 Motivation

Standard SVD/ALS factors can be negative, making them hard to interpret. **NMF** imposes the constraint that all factor values must be $$\geq 0$$, resulting in **parts-based**, naturally interpretable decompositions.

### 7.2 Formulation

Given $$R \in \mathbb{R}_{\geq 0}^{m \times n}$$, find:

$$R \approx WH, \quad W \in \mathbb{R}_{\geq 0}^{m \times k}, \quad H \in \mathbb{R}_{\geq 0}^{k \times n}$$

where:
- $$W$$ = user-topic matrix (how much each user relates to each topic)
- $$H$$ = topic-item matrix (how much each item belongs to each topic)
- $$k$$ = number of topics/components

**Objective (Frobenius norm):**

$$\min_{W, H \geq 0} \|R - WH\|_F^2 = \min_{W, H \geq 0} \sum_{i,j} \left(R_{ij} - (WH)_{ij}\right)^2$$

**KL divergence objective** (used for count data):

$$\min_{W, H \geq 0} \sum_{i,j} \left(R_{ij} \log \frac{R_{ij}}{(WH)_{ij}} - R_{ij} + (WH)_{ij}\right)$$

### 7.3 Multiplicative Update Rules (Lee & Seung, 1999)

The most widely used NMF algorithm uses multiplicative updates that guarantee non-negativity:

$$H \leftarrow H \odot \frac{W^T R}{W^T W H}$$

$$W \leftarrow W \odot \frac{R H^T}{W H H^T}$$

where $$\odot$$ denotes element-wise multiplication and $$/$$ denotes element-wise division.

### 7.4 Interpretation of Latent Components

Because all values are non-negative, each component can be interpreted as a **theme** or **topic**. This interpretability makes NMF popular in music taste discovery, e-commerce category analysis (Croma, Flipkart), and topic modeling.

### 7.5 Industrial Example: Pandora / Music Discovery

Music streaming services use NMF-style factorization to discover latent **taste profiles** (moods, energy levels, genres) that explain listening behavior. Each user is represented as a mixture of taste profiles, enabling nuanced recommendations beyond simple genre labels.

In [0]:
from sklearn.decomposition import NMF

# =============================================================================
# NMF COLLABORATIVE FILTERING
# Industrial use: Music genre/mood discovery (Pandora, Spotify)
# =============================================================================

np.random.seed(42)
n_users_nmf, n_items_nmf = 100, 30
theme_names = ['Electronic', 'Classical', 'Hip-Hop', 'Rock', 'Pop']
k_true = len(theme_names)

# Item theme assignments (soft, each item belongs primarily to one theme)
item_themes = np.zeros((n_items_nmf, k_true))
for i in range(n_items_nmf):
    primary = i % k_true
    item_themes[i, primary] = 0.8 + np.random.rand() * 0.2
    for t in range(k_true):
        if t != primary:
            item_themes[i, t] = np.random.rand() * 0.2

# User taste profiles (Dirichlet mixture of genres)
user_prefs_nmf = np.random.dirichlet(alpha=[1, 1, 1, 1, 1], size=n_users_nmf)
play_counts_true = user_prefs_nmf @ item_themes.T
play_counts_noisy = np.maximum(0, play_counts_true + np.random.randn(n_users_nmf, n_items_nmf) * 0.05)
mask_nmf = np.random.rand(n_users_nmf, n_items_nmf) > 0.75
play_counts_sparse = np.where(mask_nmf, play_counts_noisy, 0)

item_nm = [f"{theme_names[i % k_true][:3]}_Track_{i+1:02d}" for i in range(n_items_nmf)]
user_nm = [f"User_{i+1:03d}" for i in range(n_users_nmf)]
R_play = pd.DataFrame(play_counts_sparse, index=user_nm, columns=item_nm)

print("=" * 70)
print("NMF on Implicit Play Count Matrix")
print("=" * 70)
print(f"Matrix: {R_play.shape} | Sparsity: {(R_play == 0).sum().sum() / R_play.size:.1%}")

# Apply NMF
nmf_model = NMF(n_components=5, init='nndsvda', max_iter=500, random_state=42)
W = nmf_model.fit_transform(R_play.values)  # Users x Topics
H = nmf_model.components_                   # Topics x Items
print(f"NMF Reconstruction Error: {nmf_model.reconstruction_err_:.4f}")

print("\nTOP TRACKS PER DISCOVERED TOPIC")
print("-" * 60)
for t in range(5):
    top_idx = H[t].argsort()[::-1][:5]
    top_tracks = [item_nm[j] for j in top_idx]
    print(f"  Topic {t+1}: {' | '.join(top_tracks)}")

# Visualize W and H matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im1 = axes[0].imshow(W[:20], cmap='YlOrRd', aspect='auto')
axes[0].set_title('User-Topic Matrix W\n(First 20 users, 5 topics)')
axes[0].set_xlabel('Topic')
axes[0].set_ylabel('User')
axes[0].set_xticks(range(5))
axes[0].set_xticklabels([f'T{i+1}' for i in range(5)])
plt.colorbar(im1, ax=axes[0])
im2 = axes[1].imshow(H, cmap='YlGnBu', aspect='auto')
axes[1].set_title('Topic-Item Matrix H\n(5 topics x 30 tracks)')
axes[1].set_xlabel('Track Index')
axes[1].set_ylabel('Topic')
axes[1].set_yticks(range(5))
axes[1].set_yticklabels([f'Topic {i+1}' for i in range(5)])
plt.colorbar(im2, ax=axes[1])
plt.tight_layout()
display(fig)
plt.close(fig)

# Recommendations using NMF reconstruction
R_rec = W @ H
user_row = R_play.iloc[0]
unrated = [item_nm[i] for i in range(n_items_nmf) if user_row.iloc[i] == 0]
scores = {item_nm[i]: R_rec[0, i] for i in range(n_items_nmf) if user_row.iloc[i] == 0}
top5 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop-5 NMF Recommendations for User_001:")
for track, score in top5:
    print(f"  - {track}: score = {score:.4f}")

## 8. Neural Collaborative Filtering (NCF)

### 8.1 Limitations of Linear Models

All previous MF methods assume a **linear dot-product interaction** between user and item factors:

$$\hat{r}_{ui} = p_u^T q_i = \sum_{k} p_{uk} \cdot q_{ik}$$

This linear interaction misses complex, non-linear user-item relationships. **NCF** (He et al., 2017) replaces the dot product with a neural network.

### 8.2 NCF Architecture

**Input Layer:** One-hot encoded user and item IDs $$\mathbf{v}_u^U \in \{0,1\}^m, \quad \mathbf{v}_i^I \in \{0,1\}^n$$

**Embedding Layer:** Maps to dense latent vectors

$$\mathbf{p}_u = P^T \mathbf{v}_u^U, \quad \mathbf{q}_i = Q^T \mathbf{v}_i^I$$

**Neural Layers:** Multi-layer perceptron over concatenated embeddings

$$\phi_1 = \begin{bmatrix} \mathbf{p}_u \\ \mathbf{q}_i \end{bmatrix}, \quad \phi_l = \sigma\left(W_l^T \phi_{l-1} + b_l\right)$$

**Output Layer:** Sigmoid activation for implicit feedback

$$\hat{y}_{ui} = \sigma\left(\mathbf{h}^T \phi_L\right)$$

### 8.3 Generalized MF (GMF) Component

GMF restores MF as a special case of NCF:

$$\phi^{GMF} = \mathbf{p}_u^G \odot \mathbf{q}_i^G, \quad \hat{y}_{ui}^{GMF} = \sigma\left(\mathbf{h}^T (\mathbf{p}_u^G \odot \mathbf{q}_i^G)\right)$$

### 8.4 NeuMF: Unified Model

**NeuMF** combines GMF (linear) and MLP (non-linear) components:

$$\hat{y}_{ui} = \sigma\left(\mathbf{h}^T \begin{bmatrix} \phi^{GMF} \\ \phi^{MLP} \end{bmatrix}\right)$$

### 8.5 BPR Loss for Implicit Feedback

For implicit feedback, **Bayesian Personalized Ranking (BPR)** loss is used:

$$\mathcal{L}_{BPR} = -\sum_{(u,i,j) \in \mathcal{D}_S} \ln \sigma(\hat{x}_{uij}) + \lambda \|\Theta\|^2$$

where $$\hat{x}_{uij} = \hat{r}_{ui} - \hat{r}_{uj}$$ and $$j$$ is a **negative sample** (item user did not interact with). This trains the model to **rank** observed items above unobserved ones.

### 8.6 Industrial Examples

| Company | Model | Scale |
|---------|-------|-------|
| YouTube | Wide & Deep (variation of NCF) | 1B+ users, 800M videos |
| Pinterest | PinSage (Graph NCF) | 3B pins, 200M users |
| Alibaba | Deep Interest Network (DIN) | 100M+ daily active users |
| Airbnb | Embedding-based NCF | 7M listings, 150M users |

In [0]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# =============================================================================
# NeuMF: Neural Collaborative Filtering (He et al., 2017)
# Used at Pinterest, YouTube, and Alibaba at scale
# =============================================================================

class RatingsDataset(Dataset):
    """PyTorch Dataset for implicit feedback (binary: interacted=1, not=0)."""
    def __init__(self, user_ids, item_ids, labels):
        self.users  = torch.LongTensor(user_ids)
        self.items  = torch.LongTensor(item_ids)
        self.labels = torch.FloatTensor(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.users[idx], self.items[idx], self.labels[idx]


class NeuMF(nn.Module):
    """
    NeuMF (He et al., 2017) — combines:
      - GMF: element-wise product (linear MF component)
      - MLP: deep non-linear component
    Final prediction = sigmoid(h^T [phi_GMF | phi_MLP])
    """
    def __init__(self, n_users, n_items, gmf_dim=16, mlp_embed=16, mlp_layers=[64,32,16]):
        super().__init__()
        # GMF embeddings
        self.gmf_user = nn.Embedding(n_users, gmf_dim)
        self.gmf_item = nn.Embedding(n_items, gmf_dim)
        # MLP embeddings
        self.mlp_user = nn.Embedding(n_users, mlp_embed)
        self.mlp_item = nn.Embedding(n_items, mlp_embed)
        for emb in [self.gmf_user, self.gmf_item, self.mlp_user, self.mlp_item]:
            nn.init.normal_(emb.weight, std=0.01)
        # MLP layers
        layers = []
        in_sz = mlp_embed * 2
        for out_sz in mlp_layers:
            layers += [nn.Linear(in_sz, out_sz), nn.ReLU(), nn.Dropout(0.2)]
            in_sz = out_sz
        self.mlp = nn.Sequential(*layers)
        # Output projection
        self.fc = nn.Linear(gmf_dim + mlp_layers[-1], 1)
        nn.init.kaiming_uniform_(self.fc.weight)

    def forward(self, u, i):
        gmf = self.gmf_user(u) * self.gmf_item(i)               # Element-wise product
        mlp = self.mlp(torch.cat([self.mlp_user(u), self.mlp_item(i)], dim=1))
        return torch.sigmoid(self.fc(torch.cat([gmf, mlp], dim=1))).squeeze()


# --- Generate Implicit Feedback Dataset ---
np.random.seed(42)
N_USERS, N_ITEMS = 500, 200
ug = np.random.randint(0, 5, N_USERS)
ic = np.random.randint(0, 5, N_ITEMS)

pos_u, pos_i = [], []
for u in range(N_USERS):
    for i in range(N_ITEMS):
        prob = 0.08 if ug[u] == ic[i] else 0.01
        if np.random.rand() < prob:
            pos_u.append(u); pos_i.append(i)

pos_set = set(zip(pos_u, pos_i))
neg_u, neg_i = [], []
while len(neg_u) < len(pos_u) * 4:  # 4:1 negative sampling
    u, i = np.random.randint(0, N_USERS), np.random.randint(0, N_ITEMS)
    if (u, i) not in pos_set:
        neg_u.append(u); neg_i.append(i)

all_u = pos_u + neg_u
all_i = pos_i + neg_i
all_l = [1.0] * len(pos_u) + [0.0] * len(neg_u)
print(f"Positives: {len(pos_u):,} | Negatives: {len(neg_u):,} | Total: {len(all_u):,}")

split = int(0.85 * len(all_u))
train_loader = DataLoader(RatingsDataset(all_u[:split], all_i[:split], all_l[:split]),
                          batch_size=512, shuffle=True)
val_loader   = DataLoader(RatingsDataset(all_u[split:], all_i[split:], all_l[split:]),
                          batch_size=512)

# --- Train ---
dev = 'cpu'
ncf = NeuMF(N_USERS, N_ITEMS).to(dev)
opt = optim.Adam(ncf.parameters(), lr=0.001, weight_decay=1e-5)
loss_fn = nn.BCELoss()
train_hist, val_hist = [], []

print("\nTraining NeuMF...")
for epoch in range(20):
    ncf.train()
    tloss = sum(
        (loss_fn(ncf(u.to(dev), i.to(dev)), l.to(dev)), opt.zero_grad() or True,
         ncf(u.to(dev), i.to(dev)).mean().backward() or True)[0].item()
        if False else
        (lambda b: (opt.zero_grad(), loss_fn(ncf(b[0].to(dev), b[1].to(dev)), b[2].to(dev)),
                    list(map(lambda x: x.backward() if x.requires_grad else None,
                             [loss_fn(ncf(b[0].to(dev), b[1].to(dev)), b[2].to(dev))])),
                    opt.step(),
                    loss_fn(ncf(b[0].to(dev), b[1].to(dev)), b[2].to(dev)).item())[-1])(b)
        for b in train_loader
    ) if False else 0
    # Cleaner training loop
    tloss = 0
    for u, i, l in train_loader:
        opt.zero_grad()
        p = ncf(u.to(dev), i.to(dev))
        loss = loss_fn(p, l.to(dev))
        loss.backward()
        opt.step()
        tloss += loss.item()
    train_hist.append(tloss / len(train_loader))
    
    ncf.eval()
    vloss = 0
    with torch.no_grad():
        for u, i, l in val_loader:
            vloss += loss_fn(ncf(u.to(dev), i.to(dev)), l.to(dev)).item()
    val_hist.append(vloss / len(val_loader))
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/20 | Train: {train_hist[-1]:.4f} | Val: {val_hist[-1]:.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_hist, label='Train BCE', color='steelblue', linewidth=2)
ax.plot(val_hist,   label='Val BCE',   color='darkorange', linewidth=2, linestyle='--')
ax.set(xlabel='Epoch', ylabel='BCE Loss', title='NeuMF Training (He et al., 2017)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
display(fig)
plt.close(fig)
print(f"Final Val Loss: {val_hist[-1]:.4f}")

## 8B. Bayesian Personalized Ranking (BPR)

### 8B.1 The Pairwise Ranking Paradigm

All previous methods treat recommendation as a **pointwise** prediction task — predict a score for each (user, item) pair independently. But recommendation is fundamentally a **ranking** problem: we only care whether the user prefers item $$i$$ over item $$j$$, not the absolute score.

**BPR** (Rendle et al., UAI 2009) formalized this insight into a principled pairwise learning-to-rank framework for implicit feedback.

### 8B.2 Problem Setup

Given:
- $$U$$ = set of users, $$I$$ = set of items
- $$I_u^+$$ = set of items user $$u$$ has interacted with (positive feedback)
- $$I \setminus I_u^+$$ = all unobserved items (potential negatives)

We define a **personalized total order** $$>_u$$ for each user $$u$$, with the assumption:

$$i \in I_u^+ \text{ and } j \notin I_u^+ \implies i >_u j$$

i.e., observed items are preferred over unobserved items.

### 8B.3 Bayesian Derivation

BPR maximizes the **posterior probability** of the model parameters $$\Theta$$ given observed rankings:

$$p(\Theta | >_u) \propto p(>_u | \Theta) \cdot p(\Theta)$$

**Likelihood** (assuming independence across users and item pairs):

$$p(>_u | \Theta) = \prod_{(u,i,j) \in D_S} p(i >_u j | \Theta)$$

where $$D_S = \{(u, i, j) \mid u \in U, i \in I_u^+, j \in I \setminus I_u^+\}$$.

Modeling the preference probability with a sigmoid:

$$p(i >_u j | \Theta) = \sigma(\hat{x}_{uij}(\Theta))$$

where $$\hat{x}_{uij} = \hat{r}_{ui} - \hat{r}_{uj}$$ is the predicted **preference margin** and $$\sigma(x) = \frac{1}{1 + e^{-x}}$$.

**Prior:** Gaussian with zero mean (L2 regularization):

$$p(\Theta) \sim \mathcal{N}(0, \lambda^{-1} I)$$

### 8B.4 BPR-OPT: The Objective Function

Taking negative log-posterior, the **BPR optimization criterion** is:

$$\text{BPR-OPT} = \sum_{(u,i,j) \in D_S} -\ln \sigma(\hat{x}_{uij}) + \lambda_\Theta \|\Theta\|^2$$

Gradient with respect to $$\Theta$$:

$$\frac{\partial \text{BPR-OPT}}{\partial \Theta} = \sum_{(u,i,j) \in D_S} \frac{-e^{-\hat{x}_{uij}}}{1 + e^{-\hat{x}_{uij}}} \cdot \frac{\partial \hat{x}_{uij}}{\partial \Theta} + \lambda_\Theta \Theta$$

Simplified:

$$\frac{\partial \text{BPR-OPT}}{\partial \Theta} = \sum_{(u,i,j) \in D_S} -(1 - \sigma(\hat{x}_{uij})) \cdot \frac{\partial \hat{x}_{uij}}{\partial \Theta} + \lambda_\Theta \Theta$$

### 8B.5 BPR with Matrix Factorization

When the underlying model is MF:

$$\hat{r}_{ui} = p_u^T q_i$$

The preference margin becomes:

$$\hat{x}_{uij} = p_u^T q_i - p_u^T q_j = p_u^T (q_i - q_j)$$

Gradients for SGD:

$$\frac{\partial \hat{x}_{uij}}{\partial p_u} = q_i - q_j$$

$$\frac{\partial \hat{x}_{uij}}{\partial q_i} = p_u$$

$$\frac{\partial \hat{x}_{uij}}{\partial q_j} = -p_u$$

**SGD Update Rules** (with learning rate $$\alpha$$):

$$p_u \leftarrow p_u + \alpha \left[(1 - \sigma(\hat{x}_{uij}))(q_i - q_j) - \lambda p_u\right]$$

$$q_i \leftarrow q_i + \alpha \left[(1 - \sigma(\hat{x}_{uij})) \cdot p_u - \lambda q_i\right]$$

$$q_j \leftarrow q_j + \alpha \left[-(1 - \sigma(\hat{x}_{uij})) \cdot p_u - \lambda q_j\right]$$

### 8B.6 Why BPR Outperforms Pointwise Methods

| Aspect | Pointwise (BCE on 0/1) | BPR (Pairwise) |
|--------|------------------------|----------------|
| Objective | Predict absolute score | Rank items correctly |
| Negatives | Fixed 0/1 labels | Relative ordering |
| Gradient signal | From individual items | From item **pairs** |
| Calibration | Predicts interaction probability | Only preserves ranking |
| Empirical | AUC ≈ 0.78–0.82 | AUC ≈ 0.82–0.87 |

### 8B.7 Negative Sampling Strategies

Uniform random sampling of $$j$$ is simple but wasteful (most negatives are "too easy"). Advanced strategies:

- **Popularity-biased sampling**: Sample $$j$$ proportional to popularity $$\propto |U_j|^{0.75}$$
- **Hard negative mining**: Sample $$j$$ from items with high predicted scores but no interaction
- **DNS (Dynamic Negative Sampling)**: At each step, sample multiple candidates and pick the hardest
- **MixGCF**: Use graph-augmented hard negatives from GNN embeddings

### 8B.8 Industrial Examples

**Spotify — Playlist Extension (2018):**
Spotify uses BPR-MF for generating "Related Artists" and extending auto-generated playlists. The pairwise approach ensures that tracks the user has saved always rank above random tracks.

**Steam (Valve) — Game Recommendations:**
Steam's recommendation engine uses BPR to rank games: purchased games are positive, browsed-but-not-purchased are hard negatives, and random games are easy negatives.

**Zalando (Fashion E-commerce):**
Uses BPR with visual embeddings to rank fashion items — items in the user's wishlist are positives, viewed-but-not-added items are informative negatives.

In [0]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict

# =============================================================================
# BPR-MF: Bayesian Personalized Ranking with Matrix Factorization
# (Rendle et al., UAI 2009)
#
# Industrial use: Spotify playlist extension, Steam game recommendations,
# Zalando fashion ranking
# =============================================================================

class BPR_MF:
    """
    BPR-MF: Pairwise Learning-to-Rank for Implicit Feedback.
    
    Instead of predicting absolute scores, BPR learns to rank items:
    items a user interacted with should be ranked ABOVE items they didn't.
    
    Reference: Rendle, S. et al. (2009). "BPR: Bayesian Personalized Ranking
    from Implicit Feedback." UAI.
    """
    
    def __init__(self, n_users, n_items, n_factors=32, lr=0.01, reg=0.01):
        """
        Args:
            n_factors: Dimensionality of latent space (k)
            lr:        Learning rate (alpha)
            reg:       L2 regularization (lambda_Theta)
        """
        self.n_users = n_users
        self.n_items = n_items
        self.k = n_factors
        self.lr = lr
        self.reg = reg
        
        # Initialize latent factors ~ N(0, 0.01)
        np.random.seed(42)
        self.P = np.random.normal(0, 0.01, (n_users, n_factors))  # User factors
        self.Q = np.random.normal(0, 0.01, (n_items, n_factors))  # Item factors
        
        # Item biases (capture global popularity)
        self.b_i = np.zeros(n_items)
        
        self.train_auc_history = []
    
    def _predict_score(self, u, i):
        """Predict raw score for user u, item i: p_u^T q_i + b_i."""
        return self.P[u] @ self.Q[i] + self.b_i[i]
    
    def _preference_margin(self, u, i, j):
        """Compute x_uij = r_hat_ui - r_hat_uj."""
        return self._predict_score(u, i) - self._predict_score(u, j)
    
    def _sigmoid(self, x):
        """Numerically stable sigmoid."""
        return np.where(x >= 0,
                        1 / (1 + np.exp(-x)),
                        np.exp(x) / (1 + np.exp(x)))
    
    def _sample_triplet(self, user_items):
        """Sample a (user, positive_item, negative_item) triplet."""
        u = np.random.randint(self.n_users)
        while not user_items[u]:  # Skip users with no interactions
            u = np.random.randint(self.n_users)
        
        # Positive: random item from user's interactions
        i = np.random.choice(list(user_items[u]))
        
        # Negative: random item NOT in user's interactions
        j = np.random.randint(self.n_items)
        while j in user_items[u]:
            j = np.random.randint(self.n_items)
        
        return u, i, j
    
    def fit(self, interactions, n_epochs=50, n_samples_per_epoch=None, verbose=True):
        """
        Train BPR-MF using stochastic gradient descent on sampled triplets.
        
        Args:
            interactions: list of (user_id, item_id) tuples (positive feedback)
            n_epochs: Number of training epochs
            n_samples_per_epoch: Triplets sampled per epoch (default: |interactions|)
        """
        # Build user -> items lookup
        user_items = defaultdict(set)
        for u, i in interactions:
            user_items[u].add(i)
        
        if n_samples_per_epoch is None:
            n_samples_per_epoch = len(interactions)
        
        for epoch in range(n_epochs):
            epoch_loss = 0.0
            
            for _ in range(n_samples_per_epoch):
                u, i, j = self._sample_triplet(user_items)
                
                # Compute preference margin: x_uij = p_u^T(q_i - q_j) + b_i - b_j
                x_uij = self._preference_margin(u, i, j)
                
                # Gradient multiplier: (1 - sigma(x_uij))
                grad_mult = 1 - self._sigmoid(x_uij)
                
                # --- SGD Updates (BPR gradient) ---
                # User factors: p_u += lr * [(1-sig) * (q_i - q_j) - reg * p_u]
                self.P[u] += self.lr * (grad_mult * (self.Q[i] - self.Q[j]) - self.reg * self.P[u])
                
                # Positive item factors: q_i += lr * [(1-sig) * p_u - reg * q_i]
                self.Q[i] += self.lr * (grad_mult * self.P[u] - self.reg * self.Q[i])
                
                # Negative item factors: q_j += lr * [-(1-sig) * p_u - reg * q_j]
                self.Q[j] += self.lr * (-grad_mult * self.P[u] - self.reg * self.Q[j])
                
                # Item biases
                self.b_i[i] += self.lr * (grad_mult - self.reg * self.b_i[i])
                self.b_i[j] += self.lr * (-grad_mult - self.reg * self.b_i[j])
                
                # BPR loss: -ln(sigma(x_uij))
                epoch_loss += -np.log(self._sigmoid(x_uij) + 1e-10)
            
            # Estimate AUC on a sample
            auc = self._estimate_auc(user_items, n_eval=2000)
            self.train_auc_history.append(auc)
            
            if verbose and (epoch + 1) % 10 == 0:
                avg_loss = epoch_loss / n_samples_per_epoch
                print(f"  Epoch {epoch+1:3d}/{n_epochs} | BPR Loss: {avg_loss:.4f} | AUC: {auc:.4f}")
        
        return self
    
    def _estimate_auc(self, user_items, n_eval=2000):
        """Estimate AUC by sampling triplets and checking correct ordering."""
        correct = 0
        for _ in range(n_eval):
            u, i, j = self._sample_triplet(user_items)
            if self._preference_margin(u, i, j) > 0:
                correct += 1
        return correct / n_eval
    
    def rank_items(self, user_id, exclude_known=True, user_items=None):
        """Rank all items for a user (highest score first)."""
        scores = self.P[user_id] @ self.Q.T + self.b_i
        item_scores = list(enumerate(scores))
        
        if exclude_known and user_items is not None:
            item_scores = [(i, s) for i, s in item_scores if i not in user_items.get(user_id, set())]
        
        item_scores.sort(key=lambda x: x[1], reverse=True)
        return item_scores


# =============================================================================
# Generate Implicit Feedback Dataset
# Simulates a fashion e-commerce scenario (like Zalando)
# =============================================================================
np.random.seed(42)
N_USERS, N_ITEMS = 1000, 300

# 6 user segments (style preferences)
user_segments = np.random.randint(0, 6, N_USERS)
# 6 item categories (fashion styles: casual, formal, sporty, vintage, luxury, streetwear)
item_categories = np.random.randint(0, 6, N_ITEMS)
style_names = ['Casual', 'Formal', 'Sporty', 'Vintage', 'Luxury', 'Streetwear']

# Generate implicit interactions: users interact more with items in their style
interactions = []
for u in range(N_USERS):
    for i in range(N_ITEMS):
        # Higher probability if user segment matches item category
        prob = 0.12 if user_segments[u] == item_categories[i] else 0.015
        if np.random.rand() < prob:
            interactions.append((u, i))

# Build lookup for evaluation
user_items_map = defaultdict(set)
for u, i in interactions:
    user_items_map[u].add(i)

print("=" * 70)
print("BPR-MF: PAIRWISE LEARNING TO RANK")
print("=" * 70)
print(f"Users: {N_USERS} | Items: {N_ITEMS} | Interactions: {len(interactions):,}")
print(f"Density: {len(interactions)/(N_USERS*N_ITEMS):.2%}")
print(f"Avg items/user: {np.mean([len(v) for v in user_items_map.values()]):.1f}")

# --- Train BPR-MF ---
print("\nTraining BPR-MF (k=32, 50 epochs)...")
bpr = BPR_MF(N_USERS, N_ITEMS, n_factors=32, lr=0.01, reg=0.001)
bpr.fit(interactions, n_epochs=50, n_samples_per_epoch=len(interactions))

# --- Plot AUC convergence ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(bpr.train_auc_history, color='steelblue', linewidth=2)
axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random (AUC=0.5)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('AUC (estimated)')
axes[0].set_title('BPR-MF Training: AUC Convergence\n(Higher = better pairwise ranking)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.4, 1.0])

# --- Compare BPR vs Random Ranking ---
# For 100 test users, compute AUC and Precision@10
test_users = np.random.choice(list(user_items_map.keys()), size=100, replace=False)

bpr_p10, random_p10 = [], []
for u in test_users:
    known = user_items_map[u]
    # Hold out 20% of interactions for evaluation
    known_list = list(known)
    if len(known_list) < 5:
        continue
    held_out = set(known_list[int(0.8*len(known_list)):])
    
    # BPR ranking
    bpr_ranked = bpr.rank_items(u, exclude_known=False, user_items=None)
    bpr_top10 = [item for item, _ in bpr_ranked[:10]]
    bpr_p10.append(len(set(bpr_top10) & held_out) / 10)
    
    # Random ranking baseline
    random_ranked = list(np.random.permutation(N_ITEMS)[:10])
    random_p10.append(len(set(random_ranked) & held_out) / 10)

axes[1].bar(['BPR-MF', 'Random'], [np.mean(bpr_p10), np.mean(random_p10)],
            color=['steelblue', 'lightcoral'], edgecolor='black', alpha=0.8)
axes[1].set_ylabel('Precision@10')
axes[1].set_title('BPR-MF vs Random Baseline\n(Fashion E-commerce Scenario)')
axes[1].set_ylim([0, max(np.mean(bpr_p10)*1.3, 0.1)])
for i, v in enumerate([np.mean(bpr_p10), np.mean(random_p10)]):
    axes[1].text(i, v + 0.002, f"{v:.4f}", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close(fig)

# --- Show top recommendations for a sample user ---
print("\n" + "=" * 70)
print("BPR-MF RECOMMENDATIONS (Fashion E-Commerce)")
print("=" * 70)
sample_user = test_users[0]
user_style = style_names[user_segments[sample_user]]
print(f"\nUser {sample_user} (primary style: {user_style})")
print(f"  Known interactions: {len(user_items_map[sample_user])} items")

top_recs = bpr.rank_items(sample_user, exclude_known=True, user_items=user_items_map)[:10]
print(f"\n  Top-10 BPR Recommendations:")
for rank, (item_id, score) in enumerate(top_recs, 1):
    cat = style_names[item_categories[item_id]]
    match = " *" if cat == user_style else ""
    print(f"    {rank:2d}. Item {item_id:3d} [{cat:10s}] score={score:.3f}{match}")

# Count how many recommendations match user's preferred style
matching = sum(1 for item_id, _ in top_recs if item_categories[item_id] == user_segments[sample_user])
print(f"\n  Style-matched items in top-10: {matching}/10")
print("  (* = matches user's primary style preference)")
print("\n  BPR correctly learns to rank style-aligned items higher!")

## 8C. Graph Collaborative Filtering & LightGCN

### 8C.1 Motivation: CF as a Graph Problem

The user-item interaction matrix naturally defines a **bipartite graph** $$\mathcal{G} = (\mathcal{V}, \mathcal{E})$$:
- **Nodes:** $$\mathcal{V} = \mathcal{U} \cup \mathcal{I}$$ (users and items)
- **Edges:** $$(u, i) \in \mathcal{E}$$ if user $$u$$ interacted with item $$i$$

Graph Neural Networks (GNNs) exploit the **multi-hop neighborhood structure** — a user is influenced not just by items they interacted with, but by the wider collaborative network (users who liked those items, items those users liked, etc.).

### 8C.2 From GCN to LightGCN

**NGCF (Wang et al., 2019)** applied full GCN to CF:

$$\mathbf{e}_u^{(l+1)} = \sigma\left(W_1 \mathbf{e}_u^{(l)} + \sum_{i \in \mathcal{N}_u} \frac{1}{\sqrt{|\mathcal{N}_u||\mathcal{N}_i|}} (W_1 \mathbf{e}_i^{(l)} + W_2 (\mathbf{e}_i^{(l)} \odot \mathbf{e}_u^{(l)}))\right)$$

**LightGCN (He et al., SIGIR 2020)** showed that the feature transformation $$(W_1, W_2)$$ and nonlinear activation $$\sigma$$ are **unnecessary and harmful** for CF. It strips GCN to the bare essentials:

### 8C.3 LightGCN: Light Graph Convolution

The propagation rule is simply **neighborhood aggregation with symmetric normalization**:

$$\mathbf{e}_u^{(l+1)} = \sum_{i \in \mathcal{N}_u} \frac{1}{\sqrt{|\mathcal{N}_u|} \cdot \sqrt{|\mathcal{N}_i|}} \; \mathbf{e}_i^{(l)}$$

$$\mathbf{e}_i^{(l+1)} = \sum_{u \in \mathcal{N}_i} \frac{1}{\sqrt{|\mathcal{N}_i|} \cdot \sqrt{|\mathcal{N}_u|}} \; \mathbf{e}_u^{(l)}$$

where $$\mathcal{N}_u$$ = set of items user $$u$$ interacted with, and $$\mathcal{N}_i$$ = set of users who interacted with item $$i$$.

No learnable weight matrices. No nonlinear activations. No self-connections. Just pure **message passing**.

### 8C.4 Layer Combination (Final Embedding)

After $$L$$ layers of propagation, the final representation is a **weighted sum across all layers** (including the initial embedding):

$$\mathbf{e}_u = \sum_{l=0}^{L} \alpha_l \; \mathbf{e}_u^{(l)}, \qquad \mathbf{e}_i = \sum_{l=0}^{L} \alpha_l \; \mathbf{e}_i^{(l)}$$

where $$\alpha_l = \frac{1}{L+1}$$ (uniform weight, simplest and best empirically).

This multi-scale combination captures:
- $$l=0$$: raw embedding (no smoothing)
- $$l=1$$: direct neighborhood (1-hop)
- $$l=2$$: 2-hop patterns (users who liked what you liked)
- $$l=3$$: 3-hop community structure

### 8C.5 Matrix Formulation

In matrix form, let $$\mathbf{E}^{(0)} \in \mathbb{R}^{(m+n) \times d}$$ be the initial embeddings for all users and items, and let $$\tilde{A}$$ be the symmetrically normalized adjacency matrix of the bipartite graph:

$$\tilde{A} = D^{-1/2} A \; D^{-1/2}$$

where $$A = \begin{bmatrix} 0 & R \\ R^T & 0 \end{bmatrix}$$ is the $$(m+n) \times (m+n)$$ adjacency matrix, and $$D$$ is the degree diagonal.

Then propagation is:

$$\mathbf{E}^{(l+1)} = \tilde{A} \; \mathbf{E}^{(l)}$$

Final embeddings:

$$\mathbf{E} = \frac{1}{L+1} \sum_{l=0}^{L} \mathbf{E}^{(l)} = \frac{1}{L+1} \sum_{l=0}^{L} \tilde{A}^l \; \mathbf{E}^{(0)}$$

### 8C.6 Training Objective (BPR)

LightGCN is trained with BPR loss (section 8B):

$$\mathcal{L} = \sum_{(u,i,j) \in D_S} -\ln \sigma(\mathbf{e}_u^T \mathbf{e}_i - \mathbf{e}_u^T \mathbf{e}_j) + \lambda \|\mathbf{E}^{(0)}\|^2$$

Note: regularization is applied only to the **initial embeddings** $$\mathbf{E}^{(0)}$$, not the propagated ones.

### 8C.7 Why LightGCN Works

| Design Choice | Justification |
|---|---|
| No weight matrices | CF has no node features — transformations add parameters without benefit |
| No nonlinearity | Smoothing is already linear; activation breaks linearity of spectral filters |
| Layer combination | Prevents over-smoothing; low layers capture different neighborhood scales |
| Symmetric norm | Prevents high-degree nodes from dominating the aggregation |

### 8C.8 Comparison: MF vs NGCF vs LightGCN

| Model | Propagation | Parameters | Gowalla NDCG@20 | Amazon-Book NDCG@20 |
|-------|-------------|------------|-----------------|--------------------|
| MF-BPR | None (0-hop) | $$(m+n)d$$ | 0.1291 | 0.0250 |
| NGCF | Full GCN (3-hop) | $$(m+n)d + 6d^2$$ | 0.1547 | 0.0315 |
| LightGCN | Light conv (3-hop) | $$(m+n)d$$ | **0.1830** | **0.0411** |

LightGCN achieves **18–30% improvement over NGCF** with **fewer parameters** and faster training.

### 8C.9 Industrial Examples

**Pinterest — PinSage (2018):**
Pinterest deploys a graph CF model (PinSage) on 3 billion pins and 18 billion edges. Uses random-walk-based neighborhood sampling + GraphSAGE propagation for scalable GNN inference. Generates "More like this" recommendations.

**Uber Eats — Graph Learning for Food Recommendations (2020):**
Uber Eats models the (user, restaurant, dish, cuisine) graph with GNN-based CF to recommend dishes and restaurants personalized to location, time, and user history.

**Kuaishou (Chinese TikTok) — Production LightGCN (2021):**
Kuaishou deploys a multi-billion-edge LightGCN variant for short video recommendations serving 300M+ daily active users. Uses GPU-accelerated graph sampling for real-time inference.

**Twitter — TwHIN (2022):**
Twitter's heterogeneous information network uses a multi-relational graph embedding model (TwHIN) combining follows, likes, retweets, and replies for "Who to Follow" and timeline ranking.

In [0]:
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.sparse import csr_matrix, eye, diags
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# =============================================================================
# LightGCN: Simplifying and Powering Graph Convolution Network for CF
# (He et al., SIGIR 2020)
#
# Industrial use: Pinterest (PinSage), Kuaishou, Twitter (TwHIN), Uber Eats
# =============================================================================

class LightGCN:
    """
    LightGCN: Graph-based Collaborative Filtering.
    
    Key insight: Remove feature transformation & nonlinear activation from GCN.
    Only keep: neighborhood aggregation + layer combination.
    
    Reference: He, X. et al. (2020). "LightGCN: Simplifying and Powering
    Graph Convolution Network for Recommendation." SIGIR.
    """
    
    def __init__(self, n_users, n_items, embed_dim=64, n_layers=3, lr=0.001, reg=1e-4):
        """
        Args:
            embed_dim:  Embedding dimension d
            n_layers:   Number of GCN propagation layers L
            lr:         Learning rate
            reg:        L2 regularization on E^(0) only
        """
        self.n_users = n_users
        self.n_items = n_items
        self.d = embed_dim
        self.L = n_layers
        self.lr = lr
        self.reg = reg
        
        # Initialize embeddings E^(0) ~ N(0, 0.01)
        np.random.seed(42)
        self.user_embed = np.random.normal(0, 0.01, (n_users, embed_dim))  # p_u^(0)
        self.item_embed = np.random.normal(0, 0.01, (n_items, embed_dim))  # q_i^(0)
        
        self.norm_adj = None  # Will be computed from interactions
        self.history = {'loss': [], 'auc': []}
    
    def _build_norm_adj(self, interactions):
        """
        Build the symmetrically normalized adjacency matrix:
          A_tilde = D^(-1/2) * A * D^(-1/2)
        where A is the bipartite adjacency matrix.
        """
        n = self.n_users + self.n_items
        
        # Build sparse adjacency (bipartite: users 0..m-1, items m..m+n-1)
        rows, cols, data = [], [], []
        for u, i in interactions:
            rows.append(u)
            cols.append(self.n_users + i)  # Offset items
            rows.append(self.n_users + i)
            cols.append(u)
            data.extend([1.0, 1.0])
        
        A = csr_matrix((data, (rows, cols)), shape=(n, n))
        
        # Degree matrix D
        degrees = np.array(A.sum(axis=1)).flatten()
        degrees[degrees == 0] = 1  # Avoid division by zero
        D_inv_sqrt = diags(1.0 / np.sqrt(degrees))
        
        # Symmetric normalization: D^{-1/2} A D^{-1/2}
        self.norm_adj = D_inv_sqrt @ A @ D_inv_sqrt
        return self
    
    def _propagate(self):
        """
        Perform L layers of light graph convolution.
        E^(l+1) = A_tilde * E^(l)
        Final = mean(E^(0), E^(1), ..., E^(L))
        """
        # Stack all embeddings [users; items]
        E_0 = np.vstack([self.user_embed, self.item_embed])  # (m+n) x d
        
        all_layers = [E_0]
        E_l = E_0
        
        for _ in range(self.L):
            E_l = self.norm_adj @ E_l  # Graph convolution (pure message passing)
            all_layers.append(E_l)
        
        # Layer combination: uniform mean
        E_final = np.mean(all_layers, axis=0)  # (m+n) x d
        
        user_final = E_final[:self.n_users]       # m x d
        item_final = E_final[self.n_users:]        # n x d
        
        return user_final, item_final
    
    def _sigmoid(self, x):
        """Numerically stable sigmoid."""
        return np.where(x >= 0, 1/(1+np.exp(-x)), np.exp(x)/(1+np.exp(x)))
    
    def fit(self, interactions, n_epochs=30, n_samples_per_epoch=None, verbose=True):
        """
        Train with BPR loss on the propagated embeddings.
        Gradients flow back to E^(0) only.
        """
        self._build_norm_adj(interactions)
        
        user_items = defaultdict(set)
        for u, i in interactions:
            user_items[u].add(i)
        
        if n_samples_per_epoch is None:
            n_samples_per_epoch = min(len(interactions), 50000)
        
        all_users = list(user_items.keys())
        
        for epoch in range(n_epochs):
            # Forward: propagate to get final embeddings
            user_final, item_final = self._propagate()
            
            epoch_loss = 0.0
            # Stochastic BPR updates
            for _ in range(n_samples_per_epoch):
                # Sample triplet
                u = np.random.choice(all_users)
                i = np.random.choice(list(user_items[u]))
                j = np.random.randint(self.n_items)
                while j in user_items[u]:
                    j = np.random.randint(self.n_items)
                
                # Scores from propagated embeddings
                x_uij = user_final[u] @ item_final[i] - user_final[u] @ item_final[j]
                
                # BPR gradient coefficient
                coeff = 1 - self._sigmoid(x_uij)
                
                # Approximate gradient on E^(0)
                # In full LightGCN, gradients propagate through the graph.
                # Here we use a simplified direct update (efficient approximation)
                self.user_embed[u] += self.lr * (coeff * (item_final[i] - item_final[j]) - self.reg * self.user_embed[u])
                self.item_embed[i] += self.lr * (coeff * user_final[u] - self.reg * self.item_embed[i])
                self.item_embed[j] += self.lr * (-coeff * user_final[u] - self.reg * self.item_embed[j])
                
                epoch_loss += -np.log(self._sigmoid(x_uij) + 1e-10)
            
            # Estimate AUC
            auc = self._estimate_auc(user_final, item_final, user_items)
            self.history['loss'].append(epoch_loss / n_samples_per_epoch)
            self.history['auc'].append(auc)
            
            if verbose and (epoch + 1) % 5 == 0:
                print(f"  Epoch {epoch+1:2d}/{n_epochs} | Loss: {self.history['loss'][-1]:.4f} | AUC: {auc:.4f}")
        
        return self
    
    def _estimate_auc(self, user_final, item_final, user_items, n_eval=2000):
        """Estimate ranking AUC."""
        correct = 0
        all_users = list(user_items.keys())
        for _ in range(n_eval):
            u = np.random.choice(all_users)
            i = np.random.choice(list(user_items[u]))
            j = np.random.randint(self.n_items)
            while j in user_items[u]:
                j = np.random.randint(self.n_items)
            if user_final[u] @ item_final[i] > user_final[u] @ item_final[j]:
                correct += 1
        return correct / n_eval
    
    def recommend(self, user_id, user_items, top_k=10):
        """Generate top-K recommendations using propagated embeddings."""
        user_final, item_final = self._propagate()
        scores = user_final[user_id] @ item_final.T
        # Exclude known items
        known = user_items.get(user_id, set())
        for idx in known:
            scores[idx] = -np.inf
        top_items = np.argsort(scores)[::-1][:top_k]
        return [(int(idx), float(scores[idx])) for idx in top_items]


# =============================================================================
# Dataset: Social Network + Content Platform (like Twitter/Kuaishou)
# =============================================================================
np.random.seed(42)
N_USERS, N_ITEMS = 800, 400

# 5 communities with dense intra-community connections
n_communities = 5
user_community = np.random.randint(0, n_communities, N_USERS)
item_community = np.random.randint(0, n_communities, N_ITEMS)

# Generate interactions with community structure
# Users strongly prefer items in their community (models social influence)
interactions_graph = []
for u in range(N_USERS):
    for i in range(N_ITEMS):
        if user_community[u] == item_community[i]:
            prob = 0.06  # High intra-community interaction
        else:
            prob = 0.008  # Low cross-community interaction
        if np.random.rand() < prob:
            interactions_graph.append((u, i))

user_items_graph = defaultdict(set)
for u, i in interactions_graph:
    user_items_graph[u].add(i)

print("=" * 70)
print("LightGCN: GRAPH COLLABORATIVE FILTERING")
print("=" * 70)
print(f"Users: {N_USERS} | Items: {N_ITEMS} | Interactions: {len(interactions_graph):,}")
print(f"Density: {len(interactions_graph)/(N_USERS*N_ITEMS):.2%}")
print(f"Communities: {n_communities} (models social clusters)")
print(f"Avg degree (user): {np.mean([len(v) for v in user_items_graph.values()]):.1f}")

# --- Train LightGCN ---
print("\nTraining LightGCN (d=64, L=3 layers, 30 epochs)...")
lgcn = LightGCN(N_USERS, N_ITEMS, embed_dim=64, n_layers=3, lr=0.005, reg=1e-4)
lgcn.fit(interactions_graph, n_epochs=30, n_samples_per_epoch=30000)

# --- Train BPR-MF baseline for comparison ---
print("\nTraining BPR-MF baseline (k=64, 30 epochs)...")
bpr_baseline = BPR_MF(N_USERS, N_ITEMS, n_factors=64, lr=0.01, reg=0.001)
bpr_baseline.fit(interactions_graph, n_epochs=30, n_samples_per_epoch=30000, verbose=False)

print("\n" + "=" * 70)
print("RESULTS COMPARISON")
print("=" * 70)
print(f"  LightGCN (3-layer) Final AUC: {lgcn.history['auc'][-1]:.4f}")
print(f"  BPR-MF (no graph)  Final AUC: {bpr_baseline.train_auc_history[-1]:.4f}")
print(f"  Improvement: +{(lgcn.history['auc'][-1] - bpr_baseline.train_auc_history[-1])*100:.2f}% AUC")

# --- Plot comparison ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# AUC convergence comparison
axes[0].plot(lgcn.history['auc'], label='LightGCN (3 layers)', color='steelblue', linewidth=2)
axes[0].plot(bpr_baseline.train_auc_history, label='BPR-MF (no graph)', 
             color='darkorange', linewidth=2, linestyle='--')
axes[0].axhline(y=0.5, color='red', linestyle=':', alpha=0.4, label='Random')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('AUC')
axes[0].set_title('LightGCN vs BPR-MF\n(Graph structure helps!)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.45, 1.0])

# Layer contribution analysis
print("\nLayer contribution analysis...")
layer_aucs = []
for L_test in range(5):
    lgcn_test = LightGCN(N_USERS, N_ITEMS, embed_dim=64, n_layers=L_test, lr=0.005, reg=1e-4)
    lgcn_test.fit(interactions_graph, n_epochs=15, n_samples_per_epoch=20000, verbose=False)
    layer_aucs.append(lgcn_test.history['auc'][-1])

axes[1].bar(range(5), layer_aucs, color=['lightcoral', 'steelblue', 'steelblue', 'steelblue', 'lightcoral'],
            edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Number of GCN Layers (L)')
axes[1].set_ylabel('Final AUC')
axes[1].set_title('Effect of Graph Depth\n(Sweet spot: L=2-3)')
axes[1].set_xticks(range(5))
axes[1].set_xticklabels(['0\n(=MF)', '1', '2', '3', '4\n(over-smooth)'])
for idx, v in enumerate(layer_aucs):
    axes[1].text(idx, v + 0.005, f"{v:.3f}", ha='center', fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

# Community detection from embeddings
user_final, item_final = lgcn._propagate()
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
user_2d = pca.fit_transform(user_final[:200])  # First 200 users

community_colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']
for c in range(n_communities):
    mask = user_community[:200] == c
    axes[2].scatter(user_2d[mask, 0], user_2d[mask, 1], 
                    c=community_colors[c], s=20, alpha=0.6, label=f'Community {c+1}')
axes[2].set_xlabel('PC 1')
axes[2].set_ylabel('PC 2')
axes[2].set_title('LightGCN User Embeddings\n(Colors = ground-truth communities)')
axes[2].legend(markerscale=2, fontsize=8)
axes[2].grid(True, alpha=0.2)

plt.tight_layout()
display(fig)
plt.close(fig)

# --- Recommendations showcasing multi-hop reasoning ---
print("\n" + "=" * 70)
print("LightGCN MULTI-HOP RECOMMENDATIONS")
print("=" * 70)
community_names = ['Tech', 'Sports', 'Music', 'Fashion', 'Food']
test_u = 5
u_comm = community_names[user_community[test_u]]
print(f"\nUser {test_u} (community: {u_comm})")
print(f"  Known interactions: {len(user_items_graph[test_u])} items")

recs = lgcn.recommend(test_u, user_items_graph, top_k=10)
print(f"\n  Top-10 LightGCN Recommendations:")
for rank, (item_id, score) in enumerate(recs, 1):
    i_comm = community_names[item_community[item_id]]
    match = " ***" if i_comm == u_comm else ""
    print(f"    {rank:2d}. Item {item_id:3d} [{i_comm:8s}] score={score:.4f}{match}")

matching = sum(1 for item_id, _ in recs if item_community[item_id] == user_community[test_u])
print(f"\n  Community-matched items: {matching}/10")
print("  (*** = same community as user)")
print("\n  LightGCN leverages multi-hop graph structure to discover")
print("  community preferences even for items the user never directly saw!")

## 8D. WARP Loss: Weighted Approximate-Rank Pairwise

### 8D.1 Motivation: Optimizing the Top of the Ranking

BPR treats all pairwise violations equally — whether a positive item is ranked #2 or #2000, the gradient magnitude is the same. But in practice, **only the top-K positions matter** for recommendation (users never scroll past position 10–20).

**WARP** (Weston et al., ICML 2011) addresses this by weighting the loss according to the **approximate rank** of the positive item. Items ranked lower (worse) receive a **larger gradient**, focusing learning on the top of the list.

### 8D.2 WARP Loss Formulation

For a triplet $$(u, i, j)$$ where $$i \in I_u^+$$ and $$j \notin I_u^+$$:

$$\mathcal{L}_{\text{WARP}} = L(\text{rank}_u(i)) \cdot \max(0, 1 - \hat{r}_{ui} + \hat{r}_{uj})$$

where:
- $$\max(0, 1 - \hat{r}_{ui} + \hat{r}_{uj})$$ is the **hinge loss** (margin = 1)
- $$L(\text{rank}_u(i))$$ is a **rank-weighting function** that increases with rank

### 8D.3 The Rank-Weighting Function

The weighting function transforms the rank into a loss weight:

$$L(k) = \sum_{j=1}^{k} \frac{1}{j} = 1 + \frac{1}{2} + \frac{1}{3} + \cdots + \frac{1}{k}$$

This is the **harmonic series** (approximation of $$\ln(k)$$). Properties:
- If positive item $$i$$ is ranked #1 → $$L(1) = 1$$ (small weight)
- If ranked #100 → $$L(100) \approx 5.2$$ (large weight)
- If ranked #10000 → $$L(10000) \approx 9.8$$ (very large weight)

This creates a **top-heavy optimization**: the model is penalized much more for ranking a positive item low than for already having it near the top.

### 8D.4 Approximate Rank Estimation via Rejection Sampling

Computing the exact rank of item $$i$$ for user $$u$$ requires scoring all items ($$O(n)$$). WARP approximates this cheaply:

1. Sample random negative items $$j_1, j_2, \ldots$$ until finding one that **violates the margin**: $$\hat{r}_{uj_k} + 1 > \hat{r}_{ui}$$
2. If it takes $$N$$ trials to find a violator, the **approximate rank** is:

$$\widehat{\text{rank}}_u(i) \approx \left\lfloor \frac{|I| - 1}{N} \right\rfloor$$

**Intuition:**
- If $$N = 1$$ (first sample violates) → positive item is ranked poorly (many violators exist)
- If $$N$$ is large (hard to find a violator) → positive item is already ranked high

### 8D.5 WARP SGD Update

The gradient update (when a violation is found at trial $$N$$):

$$p_u \leftarrow p_u + \eta \cdot L\left(\lfloor\frac{|I|-1}{N}\rfloor\right) \cdot (q_i - q_j)$$

$$q_i \leftarrow q_i + \eta \cdot L\left(\lfloor\frac{|I|-1}{N}\rfloor\right) \cdot p_u$$

$$q_j \leftarrow q_j - \eta \cdot L\left(\lfloor\frac{|I|-1}{N}\rfloor\right) \cdot p_u$$

### 8D.6 BPR vs WARP: Detailed Comparison

| Aspect | BPR | WARP |
|--------|-----|------|
| Loss type | Log-sigmoid (smooth) | Hinge (margin-based) |
| Rank awareness | None — all pairs equal | Weighted by approx. rank |
| Optimization target | AUC (full ranking) | Precision@K (top-K) |
| Negative sampling | 1 random sample/step | Sample until violation |
| Gradient when item ranked #1 | Still receives gradient | Nearly zero gradient |
| Gradient when item ranked #1000 | Same as #1 | $$\approx 7\times$$ larger |
| Convergence | Steady, predictable | Faster for top-K, slower overall |
| Best for | Diverse recommendations | Top-K precision |
| Computational cost | $$O(1)$$ per update | $$O(1)$$ to $$O(N_{\max})$$ per update |

### 8D.7 When to Use Which

- **BPR** when optimizing overall AUC, want stable training, or have very large catalogs
- **WARP** when you care ONLY about top-5/10/20 precision (ad ranking, homepage slots, push notifications)
- **kOS-WARP** (k-Order Statistic WARP): hybrid that samples $$k$$ negatives, uses the hardest — balances cost and quality

### 8D.8 Industrial Examples

**Facebook Ads Ranking (2013–2017):**
Facebook's early ad-ranking system used WARP-style losses because only the top 3–5 ad positions generate revenue. Incorrect ranking at position #500 has zero business impact.

**LightFM Library (Lyst Fashion, 2015):**
The LightFM hybrid recommendation library (created at Lyst, a fashion marketplace) implements both BPR and WARP. Their experiments showed WARP consistently beats BPR on Precision@5 by 10–15% on fashion datasets.

**YouTube Candidate Ranking (2016):**
YouTube's deep recommendation system uses a WARP-inspired top-K objective in its ranking stage — only the top ~20 videos shown per page need to be correctly ordered.

In [0]:
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# =============================================================================
# WARP LOSS: Weighted Approximate-Rank Pairwise
# (Weston et al., ICML 2011)
#
# Industrial use: Facebook Ads, LightFM (Lyst), YouTube candidate ranking
# =============================================================================

class WARP_MF:
    """
    WARP-MF: Rank-weighted pairwise learning.
    
    Key difference from BPR: The gradient is WEIGHTED by the approximate
    rank of the positive item. Items ranked lower get proportionally larger
    gradients, focusing optimization on the top of the ranking.
    
    Reference: Weston, J. et al. (2011). "WSABIE: Scaling Up to Large
    Vocabulary Image Annotation." IJCAI.
    """
    
    def __init__(self, n_users, n_items, n_factors=32, lr=0.01, reg=0.001, max_trials=50):
        """
        Args:
            max_trials: Max sampling attempts to find a violating negative.
                        Controls trade-off between rank approx accuracy and speed.
        """
        self.n_users = n_users
        self.n_items = n_items
        self.k = n_factors
        self.lr = lr
        self.reg = reg
        self.max_trials = max_trials
        
        np.random.seed(42)
        self.P = np.random.normal(0, 0.01, (n_users, n_factors))
        self.Q = np.random.normal(0, 0.01, (n_items, n_factors))
        
        # Precompute rank weights L(k) = sum(1/j for j=1..k)
        self._rank_weights = np.zeros(n_items + 1)
        for k in range(1, n_items + 1):
            self._rank_weights[k] = self._rank_weights[k-1] + 1.0 / k
        
        self.history = {'precision_at_5': [], 'precision_at_10': [], 'auc': []}
    
    def _score(self, u, i):
        return self.P[u] @ self.Q[i]
    
    def _approximate_rank(self, u, i, user_items):
        """
        WARP's rejection sampling to approximate rank.
        Sample negatives until finding one that violates the margin.
        
        Returns: (violating_item_j, approximate_rank)
        """
        score_i = self._score(u, i)
        
        for n_trial in range(1, self.max_trials + 1):
            j = np.random.randint(self.n_items)
            while j in user_items[u]:
                j = np.random.randint(self.n_items)
            
            score_j = self._score(u, j)
            
            # Check margin violation: score_j + 1 > score_i
            if score_j + 1.0 > score_i:
                # Found a violator after n_trial attempts
                approx_rank = int((self.n_items - 1) / n_trial)
                return j, approx_rank, n_trial
        
        # No violation found (item already ranked high)
        return None, 1, self.max_trials
    
    def fit(self, interactions, n_epochs=50, n_samples_per_epoch=None, verbose=True):
        """Train WARP-MF."""
        user_items = defaultdict(set)
        for u, i in interactions:
            user_items[u].add(i)
        
        all_users_with_items = [u for u in user_items if len(user_items[u]) > 0]
        
        if n_samples_per_epoch is None:
            n_samples_per_epoch = len(interactions)
        
        for epoch in range(n_epochs):
            n_updates = 0
            
            for _ in range(n_samples_per_epoch):
                # Sample user and positive item
                u = np.random.choice(all_users_with_items)
                i = np.random.choice(list(user_items[u]))
                
                # Approximate rank via rejection sampling
                result = self._approximate_rank(u, i, user_items)
                j, approx_rank, n_trials = result
                
                if j is None:
                    continue  # Item already well-ranked, skip
                
                # Rank weight: L(approx_rank)
                weight = self._rank_weights[min(approx_rank, self.n_items)]
                
                # Gradient update (hinge loss with rank weighting)
                # Loss = L(rank) * max(0, 1 - score_i + score_j)
                self.P[u] += self.lr * weight * (self.Q[i] - self.Q[j]) - self.lr * self.reg * self.P[u]
                self.Q[i] += self.lr * weight * self.P[u] - self.lr * self.reg * self.Q[i]
                self.Q[j] -= self.lr * weight * self.P[u] + self.lr * self.reg * self.Q[j]
                n_updates += 1
            
            # Track metrics
            auc, p5, p10 = self._evaluate(user_items, n_eval_users=50)
            self.history['auc'].append(auc)
            self.history['precision_at_5'].append(p5)
            self.history['precision_at_10'].append(p10)
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1:3d}/{n_epochs} | P@5: {p5:.4f} | P@10: {p10:.4f} | AUC: {auc:.4f} | Updates: {n_updates:,}")
        
        return self
    
    def _evaluate(self, user_items, n_eval_users=50):
        """Evaluate AUC, Precision@5, Precision@10."""
        all_users = [u for u in user_items if len(user_items[u]) >= 5]
        eval_users = np.random.choice(all_users, size=min(n_eval_users, len(all_users)), replace=False)
        
        p5_list, p10_list = [], []
        auc_correct = 0
        auc_total = 0
        
        for u in eval_users:
            known = list(user_items[u])
            # Use last 20% as held-out "test" items
            split = max(1, int(0.8 * len(known)))
            test_items = set(known[split:])
            
            if not test_items:
                continue
            
            # Score all items
            scores = self.P[u] @ self.Q.T
            ranked = np.argsort(scores)[::-1]
            
            # Precision@K
            top5  = set(ranked[:5])
            top10 = set(ranked[:10])
            p5_list.append(len(top5 & test_items) / 5)
            p10_list.append(len(top10 & test_items) / 10)
            
            # AUC sampling
            for _ in range(20):
                pos = np.random.choice(list(test_items))
                neg = np.random.randint(self.n_items)
                while neg in user_items[u]:
                    neg = np.random.randint(self.n_items)
                if scores[pos] > scores[neg]:
                    auc_correct += 1
                auc_total += 1
        
        auc = auc_correct / max(auc_total, 1)
        return auc, np.mean(p5_list) if p5_list else 0, np.mean(p10_list) if p10_list else 0


# =============================================================================
# HEAD-TO-HEAD: WARP vs BPR on the same dataset
# =============================================================================
np.random.seed(42)
N_U, N_I = 800, 250

# 5 user clusters, 5 item categories (e-commerce product categories)
user_cluster = np.random.randint(0, 5, N_U)
item_cluster = np.random.randint(0, 5, N_I)
category_names = ['Electronics', 'Fashion', 'Home', 'Sports', 'Books']

# Generate interactions with cluster affinity
interactions_warp = []
for u in range(N_U):
    for i in range(N_I):
        prob = 0.10 if user_cluster[u] == item_cluster[i] else 0.012
        if np.random.rand() < prob:
            interactions_warp.append((u, i))

print("=" * 70)
print("WARP vs BPR: HEAD-TO-HEAD COMPARISON")
print("=" * 70)
print(f"Users: {N_U} | Items: {N_I} | Interactions: {len(interactions_warp):,}")
print(f"Density: {len(interactions_warp)/(N_U*N_I):.2%}")

# --- Train WARP ---
print("\n--- Training WARP-MF (k=32, 40 epochs) ---")
warp_model = WARP_MF(N_U, N_I, n_factors=32, lr=0.01, reg=0.001, max_trials=30)
warp_model.fit(interactions_warp, n_epochs=40, n_samples_per_epoch=len(interactions_warp))

# --- Train BPR (reuse class from earlier cell) ---
print("\n--- Training BPR-MF (k=32, 40 epochs) ---")
bpr_compare = BPR_MF(N_U, N_I, n_factors=32, lr=0.01, reg=0.001)
bpr_compare.fit(interactions_warp, n_epochs=40, n_samples_per_epoch=len(interactions_warp), verbose=True)

# Evaluate BPR on same metrics
user_items_warp = defaultdict(set)
for u, i in interactions_warp:
    user_items_warp[u].add(i)

# Compute P@5, P@10 for BPR at final epoch
def eval_bpr_topk(bpr_model, user_items, n_eval=50):
    all_users = [u for u in user_items if len(user_items[u]) >= 5]
    eval_users = np.random.choice(all_users, size=min(n_eval, len(all_users)), replace=False)
    p5, p10 = [], []
    for u in eval_users:
        known = list(user_items[u])
        split = max(1, int(0.8 * len(known)))
        test = set(known[split:])
        if not test:
            continue
        scores = bpr_model.P[u] @ bpr_model.Q.T + bpr_model.b_i
        ranked = np.argsort(scores)[::-1]
        p5.append(len(set(ranked[:5]) & test) / 5)
        p10.append(len(set(ranked[:10]) & test) / 10)
    return np.mean(p5), np.mean(p10)

bpr_p5, bpr_p10 = eval_bpr_topk(bpr_compare, user_items_warp)

# --- Comparison Plot ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. AUC convergence
axes[0].plot(warp_model.history['auc'], label='WARP', color='steelblue', linewidth=2)
axes[0].plot(bpr_compare.train_auc_history, label='BPR', color='darkorange', linewidth=2, linestyle='--')
axes[0].axhline(y=0.5, color='red', linestyle=':', alpha=0.4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('AUC')
axes[0].set_title('AUC Convergence\n(BPR optimizes for this)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.4, 1.0])

# 2. Precision@5 over epochs (WARP's strength)
axes[1].plot(warp_model.history['precision_at_5'], label='WARP P@5', color='steelblue', linewidth=2)
axes[1].plot(warp_model.history['precision_at_10'], label='WARP P@10', color='steelblue', linewidth=1.5, linestyle='-.')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Precision')
axes[1].set_title('WARP Precision@K Convergence\n(WARP optimizes for this)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Final comparison bar chart
metrics = ['P@5', 'P@10', 'AUC']
warp_scores = [
    warp_model.history['precision_at_5'][-1],
    warp_model.history['precision_at_10'][-1],
    warp_model.history['auc'][-1]
]
bpr_scores = [bpr_p5, bpr_p10, bpr_compare.train_auc_history[-1]]

x = np.arange(len(metrics))
width = 0.35
bars1 = axes[2].bar(x - width/2, warp_scores, width, label='WARP', color='steelblue', edgecolor='black', alpha=0.8)
bars2 = axes[2].bar(x + width/2, bpr_scores, width, label='BPR', color='darkorange', edgecolor='black', alpha=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics)
axes[2].set_ylabel('Score')
axes[2].set_title('Final Metrics: WARP vs BPR\n(Different strengths!)')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

for bar in bars1 + bars2:
    h = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2., h + 0.005, f'{h:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
display(fig)
plt.close(fig)

# --- Summary ---
print("\n" + "=" * 70)
print("FINAL COMPARISON SUMMARY")
print("=" * 70)
print(f"{'Metric':<15} {'WARP':>10} {'BPR':>10} {'Winner':>10}")
print("-" * 50)
for metric, w, b in zip(metrics, warp_scores, bpr_scores):
    winner = 'WARP' if w > b else 'BPR'
    print(f"{metric:<15} {w:>10.4f} {b:>10.4f} {winner:>10}")

print("\n" + "=" * 70)
print("KEY INSIGHT")
print("=" * 70)
print("""
  WARP optimizes for TOP-K PRECISION (what users actually see).
  BPR optimizes for overall AUC (full ranking quality).

  Practical guidance:
  - Homepage slots, push notifications, email recs -> WARP
  - "Explore" feeds, full catalog browsing -> BPR
  - Modern practice: Use WARP for final ranking, BPR for candidate generation

  The rank-weighting mechanism in WARP (L(rank) = H_k harmonic series)
  ensures the model STOPS learning once positive items are at the top,
  focusing all gradient budget on truly misranked items.
""")

## 9. Evaluation Metrics for Recommender Systems

### 9.1 Rating Prediction Metrics (Explicit Feedback)

For systems that predict numerical ratings:

**Root Mean Squared Error (RMSE):**

$$\text{RMSE} = \sqrt{\frac{1}{|\mathcal{T}|} \sum_{(u,i) \in \mathcal{T}} (r_{ui} - \hat{r}_{ui})^2}$$

RMSE penalizes large errors more heavily due to squaring (used by Netflix Prize).

**Mean Absolute Error (MAE):**

$$\text{MAE} = \frac{1}{|\mathcal{T}|} \sum_{(u,i) \in \mathcal{T}} |r_{ui} - \hat{r}_{ui}|$$

MAE treats all errors equally. More interpretable ("off by 0.5 stars on average").

### 9.2 Ranking Metrics (Top-K Recommendation)

For systems that generate top-K recommendation lists:

**Precision@K:**

$$\text{Precision@K} = \frac{\text{\# relevant items in top-}K}{K}$$

**Recall@K:**

$$\text{Recall@K} = \frac{\text{\# relevant items in top-}K}{\text{total \# relevant items}}$$

**F1@K:**

$$\text{F1@K} = \frac{2 \cdot \text{Precision@K} \cdot \text{Recall@K}}{\text{Precision@K} + \text{Recall@K}}$$

**Average Precision@K (AP@K):**

$$\text{AP@K} = \frac{1}{\min(K, |\text{rel}|)} \sum_{k=1}^{K} \text{Precision@k} \cdot \mathbb{1}[\text{item}_k \text{ is relevant}]$$

**Mean Average Precision (MAP@K):**

$$\text{MAP@K} = \frac{1}{|U|} \sum_{u=1}^{|U|} \text{AP@K}(u)$$

### 9.3 Normalized Discounted Cumulative Gain (NDCG@K)

NDCG accounts for **position bias** — relevant items ranked higher should get more credit:

**DCG@K:**

$$\text{DCG@K} = \sum_{k=1}^{K} \frac{\text{rel}_k}{\log_2(k+1)}$$

**Ideal DCG (IDCG@K):** DCG of the perfect ranking.

**NDCG@K:**

$$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}} \in [0, 1]$$

NDCG@10 is the **standard metric** in industry (Google, Amazon, Netflix).

### 9.4 Coverage and Diversity Metrics

**Catalog Coverage:**

$$\text{Coverage} = \frac{|\bigcup_{u} \text{Rec}(u)|}{|I|} $$

Measures what fraction of the item catalog is ever recommended.

**Intra-List Diversity (ILD):**

$$\text{ILD}(L) = \frac{1}{|L|(|L|-1)} \sum_{i,j \in L, i \neq j} (1 - \text{sim}(i, j))$$

**Novelty:** Average popularity of recommended items (lower popularity = more novel).

### 9.5 Business Metrics vs. Research Metrics

| Research Metric | Business Metric |
|---|---|
| RMSE, NDCG | Click-Through Rate (CTR) |
| Coverage | Revenue per session |
| Diversity | Dwell time / engagement |
| Novelty | Retention / repeat visits |

In [0]:
# =============================================================================
# COMPREHENSIVE EVALUATION METRICS FOR RECOMMENDER SYSTEMS
# Industry standard: NDCG@K, MAP@K, Precision@K, Recall@K
# =============================================================================

def rmse(y_true, y_pred):
    """Root Mean Squared Error."""
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def mae(y_true, y_pred):
    """Mean Absolute Error."""
    return np.mean(np.abs(np.array(y_true) - np.array(y_pred)))

def precision_at_k(recommended, relevant, k):
    """Precision@K: fraction of top-K that are relevant."""
    top_k = recommended[:k]
    return len(set(top_k) & set(relevant)) / k

def recall_at_k(recommended, relevant, k):
    """Recall@K: fraction of relevant items retrieved in top-K."""
    if not relevant:
        return 0.0
    top_k = recommended[:k]
    return len(set(top_k) & set(relevant)) / len(relevant)

def average_precision_at_k(recommended, relevant, k):
    """AP@K: precision averaged over positions where relevant items appear."""
    if not relevant:
        return 0.0
    score, num_hits = 0.0, 0
    for idx, item in enumerate(recommended[:k]):
        if item in relevant:
            num_hits += 1
            score += num_hits / (idx + 1)  # Precision at this position
    return score / min(len(relevant), k)

def dcg_at_k(recommended, relevant_scores, k):
    """DCG@K: sum(rel_k / log2(k+1))."""
    score = 0.0
    for idx, item in enumerate(recommended[:k]):
        rel = relevant_scores.get(item, 0)
        score += rel / np.log2(idx + 2)  # log2(rank+1), rank is 1-indexed
    return score

def ndcg_at_k(recommended, relevant_scores, k):
    """NDCG@K: DCG@K normalized by ideal DCG."""
    ideal_ranking = sorted(relevant_scores.keys(), key=lambda x: relevant_scores[x], reverse=True)
    idcg = dcg_at_k(ideal_ranking, relevant_scores, k)
    if idcg == 0:
        return 0.0
    return dcg_at_k(recommended, relevant_scores, k) / idcg

def mean_average_precision(users_recs, users_relevant, k):
    """MAP@K averaged over all users."""
    return np.mean([average_precision_at_k(users_recs[u], users_relevant[u], k)
                    for u in users_recs])

def catalog_coverage(users_recs, n_items, k):
    """Fraction of catalog covered by recommendations."""
    all_recs = set(item for recs in users_recs.values() for item in recs[:k])
    return len(all_recs) / n_items


# --- Simulate a recommendation scenario ---
np.random.seed(42)
n_eval_users = 100
n_eval_items = 500

# Generate ground truth (items each user actually liked)
relevant_items = {u: set(np.random.choice(n_eval_items, size=np.random.randint(5, 20), replace=False))
                  for u in range(n_eval_users)}

# Simulate 3 different models' top-20 recommendations
def simulate_recommendations(relevant, n_items, precision_rate, n_users, k=20, seed=42):
    """Simulate a model with given precision quality."""
    np.random.seed(seed)
    recs = {}
    for u in range(n_users):
        rel = list(relevant[u])
        n_correct = int(k * precision_rate)  # How many correct items included
        correct = np.random.choice(rel, size=min(n_correct, len(rel)), replace=False).tolist()
        noise = np.random.choice([i for i in range(n_items) if i not in relevant[u]],
                                 size=k - len(correct), replace=False).tolist()
        shuffled = correct + noise
        np.random.shuffle(shuffled)
        recs[u] = shuffled[:k]
    return recs

recs_good   = simulate_recommendations(relevant_items, n_eval_items, 0.5, n_eval_users, seed=1)
recs_medium = simulate_recommendations(relevant_items, n_eval_items, 0.25, n_eval_users, seed=2)
recs_random = simulate_recommendations(relevant_items, n_eval_items, 0.1, n_eval_users, seed=3)

# Compute metrics for K = 5, 10, 20
print("=" * 75)
print("RECOMMENDER SYSTEM EVALUATION RESULTS (100 users, 500 items)")
print("=" * 75)

models = {
    'Model A (Good)':   recs_good,
    'Model B (Medium)': recs_medium,
    'Model C (Random)': recs_random,
}

for k in [5, 10, 20]:
    print(f"\n--- K = {k} ---")
    rows = []
    for name, recs in models.items():
        p = np.mean([precision_at_k(recs[u], relevant_items[u], k) for u in recs])
        r = np.mean([recall_at_k(recs[u], relevant_items[u], k) for u in recs])
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        map_k = mean_average_precision(recs, relevant_items, k)
        # For NDCG, use binary relevance
        rel_scores = {u: {item: 1.0 for item in relevant_items[u]} for u in recs}
        ndcg = np.mean([ndcg_at_k(recs[u], rel_scores[u], k) for u in recs])
        cov = catalog_coverage(recs, n_eval_items, k)
        rows.append({'Model': name, f'P@{k}': round(p,4), f'R@{k}': round(r,4),
                     f'F1@{k}': round(f1,4), f'MAP@{k}': round(map_k,4),
                     f'NDCG@{k}': round(ndcg,4), 'Coverage': f"{cov:.1%}"})
    display(pd.DataFrame(rows).set_index('Model'))

# --- NDCG decomposition: position matters ---
print("\n" + "=" * 75)
print("NDCG POSITION SENSITIVITY: Same relevance, different rank positions")
print("=" * 75)
relevant_ex = {0, 1, 2}  # 3 relevant items
rel_scores_ex = {0: 1.0, 1: 1.0, 2: 1.0}

scenarios = [
    ([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], "All relevant items ranked first"),
    ([3, 0, 1, 2, 4, 5, 6, 7, 8, 9], "Relevant at positions 2,3,4"),
    ([3, 4, 5, 0, 1, 2, 6, 7, 8, 9], "Relevant at positions 4,5,6"),
    ([3, 4, 5, 6, 7, 8, 0, 1, 2, 9], "Relevant at positions 7,8,9"),
]
for rec, desc in scenarios:
    ndcg = ndcg_at_k(rec, rel_scores_ex, 10)
    prec = precision_at_k(rec, relevant_ex, 10)
    print(f"  {desc:<40} | P@10 = {prec:.2f} | NDCG@10 = {ndcg:.4f}")
print("\n=> Precision@10 is same for all, but NDCG penalizes lower-ranked relevant items.")

## 10. The Cold Start Problem & Hybrid Approaches

### 10.1 Cold Start Variants

The cold start problem arises when there is insufficient historical data:

| Type | Description | Affected Algorithms |
|------|-------------|--------------------|
| **New User** | User has no ratings history | All CF methods |
| **New Item** | Item has no ratings | User-Based CF, ALS |
| **New System** | Platform just launched | All CF methods |

### 10.2 Solutions

#### 10.2.1 Content-Based Bootstrapping

For new items: use item features (genre, description, image) to estimate initial factor vectors.

$$q_i^{\text{new}} = \arg\min_{q} \|q - \phi(x_i)\|^2$$

where $$\phi(x_i)$$ is a feature embedding of the new item's attributes.

#### 10.2.2 Onboarding / Active Learning

For new users: ask targeted questions to gather initial preferences quickly.

Optimal item selection strategy: choose the item that **maximizes information gain**:

$$i^* = \arg\max_{i} H(p_u) - \mathbb{E}_{r}[H(p_u | r_{ui} = r)]$$

#### 10.2.3 Popularity-Based Fallback

Simplest approach: recommend globally popular items when user history is unavailable.

$$\hat{r}_{ui}^{\text{fallback}} = \bar{r}_i + \epsilon$$

#### 10.2.4 Hybrid Models

**Linear Combination:**

$$\hat{r}_{ui}^{\text{hybrid}} = \alpha \cdot \hat{r}_{ui}^{\text{CF}} + (1-\alpha) \cdot \hat{r}_{ui}^{\text{CB}}$$

where $$\alpha$$ can be learned or set as a function of the user's interaction count:

$$\alpha(n_u) = \frac{n_u}{n_u + \beta}$$

As $$n_u \to \infty$$, $$\alpha \to 1$$ (full CF). As $$n_u \to 0$$, $$\alpha \to 0$$ (full content-based).

**Switching Strategy:** Use CF if user has $$\geq T$$ interactions, else use content-based.

### 10.3 Industrial Examples

**YouTube (Cold Start for new videos):**
- New videos bootstrapped with their title/description embeddings
- Promoted to exploratory traffic for quick feedback collection
- Transitioned to CF-based ranking once 100+ interactions accumulate

**Spotify (New Artist Cold Start):**
- Artist's audio features (tempo, key, energy from Echo Nest)
- Compared to audio signatures of established artists
- Initial placement in playlists with similar acoustic profiles

**Netflix (New User Cold Start):**
- Onboarding screen asking users to rate 10 title cards
- Genre preference selection shown after signup
- Transitions to full CF after ~20 ratings

In [0]:
# =============================================================================
# HYBRID RECOMMENDER: Adaptive blend of CF + Content-Based
# Demonstrates cold start handling with smooth alpha transition
# =============================================================================

class AdaptiveHybridRecommender:
    """
    Adaptive hybrid: blends CF and content-based predictions.
    
    Cold start handling strategy:
      alpha(n) = n / (n + beta)
      - New user (n=0):  alpha=0  -> 100% content-based
      - n=beta:          alpha=0.5 -> 50/50 blend
      - Expert user:     alpha->1  -> 100% CF
    """
    
    def __init__(self, beta=10, cf_model=None, item_features=None):
        """
        beta: controls transition speed (larger = slower to trust CF).
        """
        self.beta = beta
        self.cf_model = cf_model
        self.item_features = item_features  # n_items x n_features
    
    def _alpha(self, n_interactions):
        """Compute CF weight based on interaction count."""
        return n_interactions / (n_interactions + self.beta)
    
    def _content_score(self, user_feature_profile, item_idx):
        """Simple cosine similarity between user taste vector and item features."""
        if user_feature_profile is None:
            return 0.5  # neutral
        u = user_feature_profile
        i = self.item_features[item_idx]
        denom = (np.linalg.norm(u) * np.linalg.norm(i))
        return float(np.dot(u, i) / denom) if denom > 0 else 0.5
    
    def predict(self, user_idx, item_idx, n_interactions, user_feature_profile=None):
        """Hybrid prediction with adaptive blending."""
        alpha = self._alpha(n_interactions)
        
        # CF score (from latent factor model)
        cf_score = self.cf_model.predict(user_idx, item_idx) / 5.0  # normalize to [0,1]
        
        # Content-based score
        cb_score = self._content_score(user_feature_profile, item_idx)
        
        # Weighted blend
        hybrid = alpha * cf_score + (1 - alpha) * cb_score
        return hybrid, alpha, cf_score, cb_score


# --- Demonstrate cold start transition ---
np.random.seed(42)

# Reuse the Funk SVD model from earlier
item_features_sim = np.random.rand(n_items, 5)  # 5-dim content features per item
item_features_sim = item_features_sim / item_features_sim.sum(axis=1, keepdims=True)

hybrid = AdaptiveHybridRecommender(beta=10, cf_model=funk_svd, item_features=item_features_sim)

# Simulate a new user accumulating interactions over time
user_idx_test = 5
item_to_predict = 15
user_taste = np.array([0.6, 0.2, 0.1, 0.05, 0.05])  # Strong preference for feature 0

print("=" * 70)
print("COLD START TRANSITION: New User Accumulates Interactions")
print("=" * 70)
print(f"{'Interactions':>15} | {'alpha (CF weight)':>18} | {'CF Score':>10} | {'CB Score':>10} | {'Hybrid':>10}")
print("-" * 70)

interaction_counts = [0, 1, 2, 5, 10, 20, 50, 100, 200]
alphas, hybrids, cf_scores_list, cb_scores_list = [], [], [], []

for n in interaction_counts:
    hybrid_score, alpha, cf_s, cb_s = hybrid.predict(
        user_idx_test, item_to_predict, n, user_taste
    )
    alphas.append(alpha)
    hybrids.append(hybrid_score)
    cf_scores_list.append(cf_s)
    cb_scores_list.append(cb_s)
    print(f"{n:>15} | {alpha:>18.4f} | {cf_s:>10.4f} | {cb_s:>10.4f} | {hybrid_score:>10.4f}")

# Visualize the transition
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(interaction_counts, alphas, 'o-', color='steelblue', linewidth=2)
axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='50/50 blend')
axes[0].set_xlabel('Number of User Interactions')
axes[0].set_ylabel(r'$\alpha$ (CF weight)')
axes[0].set_title(r'Cold Start Transition: $\alpha(n) = n/(n+\beta)$, $\beta$=10')
axes[0].set_xscale('log')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([-0.05, 1.05])

axes[1].plot(interaction_counts, hybrids,       'o-', label='Hybrid Score',       color='purple',   linewidth=2)
axes[1].plot(interaction_counts, cf_scores_list, 's--', label='CF Score (pure)',    color='steelblue', linewidth=1.5)
axes[1].plot(interaction_counts, cb_scores_list, '^--', label='Content-Based Score', color='darkorange', linewidth=1.5)
axes[1].set_xlabel('Number of User Interactions')
axes[1].set_ylabel('Predicted Score (normalized)')
axes[1].set_title('Score Components vs. Interaction History')
axes[1].set_xscale('log')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
display(fig)
plt.close(fig)
print("\nAs interactions grow, the model smoothly transitions from content-based to CF.")

## 11. Summary & Algorithm Comparison

### 11.1 Quick Reference: When to Use What

| Algorithm | Best For | Avoid When | Key Hyperparameter | Typical RMSE |
|-----------|----------|------------|--------------------|--------------|
| User-Based CF | Small datasets, high interpretability needs | >1M users | k neighbors | 0.9–1.1 |
| Item-Based CF | Stable item catalog, offline precompute | Catalog changes rapidly | k neighbors | 0.85–1.0 |
| Funk SVD | Explicit ratings, sequential updates | Distributed training needed | k factors, lr, reg | 0.85–0.90 |
| ALS (Spark) | Implicit feedback at scale, distributed | Very small datasets | rank, maxIter, reg | 0.80–0.90 |
| NMF | Interpretable topics, non-negative data | Negative patterns exist | k components | 0.88–0.95 |
| NeuMF | Large-scale, complex non-linear patterns | Limited compute | embed dim, MLP layers | 0.75–0.85 |

### 11.2 Complexity Summary

| Algorithm | Training Time | Prediction Time | Space |
|-----------|--------------|-----------------|-------|
| User-Based CF | $$O(m^2 \cdot n)$$ | $$O(k \cdot n)$$ | $$O(m^2)$$ |
| Item-Based CF | $$O(n^2 \cdot m)$$ | $$O(k)$$ | $$O(n^2)$$ |
| SVD/MF (SGD) | $$O(T \cdot |\mathcal{K}| \cdot k)$$ | $$O(k)$$ | $$O((m+n)k)$$ |
| ALS | $$O(I \cdot |\mathcal{K}| \cdot k^2)$$ | $$O(k)$$ | $$O((m+n)k)$$ |
| NMF | $$O(T \cdot m \cdot n \cdot k)$$ | $$O(k)$$ | $$O((m+n)k)$$ |
| NeuMF | $$O(T \cdot |\mathcal{K}| \cdot d^2)$$ | $$O(d^2 L)$$ | $$O((m+n)d)$$ |

where $$T$$ = epochs, $$|\mathcal{K}|$$ = observed ratings, $$k$$ = factors, $$d$$ = embedding dim, $$L$$ = MLP depth.

### 11.3 Evolution of CF in Industry

```
1992  Tapestry (Goldberg et al.) — coined "collaborative filtering"
1994  GroupLens — Usenet news, first User-Based CF at scale
2001  Amazon item-to-item CF (patented) — $1B+ revenue impact
2006  Netflix Prize — Funk SVD, matrix factorization era begins
2008  SVD++ (Koren) — implicit feedback integration
2008  Implicit ALS (Hu, Koren, Volinsky) — Spotify's algorithm
2016  Wide & Deep (Google) — neural + memorization hybrid
2017  NeuMF (He et al.) — neural CF becomes standard
2019  BERT4Rec (Sun et al.) — transformer-based sequential CF
2021  Graph CF (LightGCN) — graph convolution on user-item bipartite graph
2023  Diffusion-based Recommenders — generative approach
```

### 11.4 Key Takeaways

1. **Start simple**: Item-Based CF is often surprisingly competitive and highly interpretable.
2. **Scale with ALS**: When your data exceeds memory, Spark MLlib ALS is the standard production choice.
3. **Implicit > Explicit**: In most real systems, clicks/purchases vastly outnumber explicit ratings.
4. **Evaluate correctly**: NDCG@K or MAP@K over RMSE for production recommendation lists.
5. **Cold start is always there**: Build hybrid systems from the start, not as an afterthought.
6. **Neural CF is powerful but expensive**: Use NeuMF when you have GPU compute and large-scale data.
7. **Business metrics are ground truth**: Offline RMSE improvements don't always translate to online CTR gains.

## 12. References & Further Reading

---

### 12.1 Foundational Papers

| Year | Paper | Contribution |
|------|-------|-------------|
| 1992 | Goldberg, D. et al. "Using Collaborative Filtering to Weave an Information Tapestry." *CACM* 35(12). | Coined the term "collaborative filtering" |
| 1994 | Resnick, P. et al. "GroupLens: An Open Architecture for Collaborative Filtering." *CSCW*. | First scalable User-Based CF system |
| 2001 | Sarwar, B. et al. "Item-Based Collaborative Filtering Recommendation Algorithms." *WWW*. | Introduced Item-Based CF |
| 2003 | Linden, G., Smith, B. & York, J. "Amazon.com Recommendations: Item-to-Item Collaborative Filtering." *IEEE Internet Computing*. | Amazon's production system |
| 2009 | Koren, Y., Bell, R. & Volinsky, C. "Matrix Factorization Techniques for Recommender Systems." *IEEE Computer* 42(8). | Netflix Prize overview; SVD++, biases |

### 12.2 Model-Based & Latent Factor Methods

| Year | Paper | Contribution |
|------|-------|-------------|
| 2006 | Funk, S. "Netflix Update: Try This at Home." (Blog post) | Funk SVD with SGD |
| 2008 | Hu, Y., Koren, Y. & Volinsky, C. "Collaborative Filtering for Implicit Feedback Datasets." *ICDM*. | Implicit ALS (Spotify's algorithm) |
| 2008 | Koren, Y. "Factorization Meets the Neighborhood." *KDD*. | SVD++ combining latent factors + neighborhood |
| 2012 | Zhou, Y. et al. "Large-Scale Parallel Collaborative Filtering for the Netflix Prize." *AAIM*. | ALS at scale |
| 1999 | Lee, D. & Seung, H. "Learning the Parts of Objects by Non-Negative Matrix Factorization." *Nature* 401. | NMF foundations |

### 12.3 Pairwise Learning & Ranking

| Year | Paper | Contribution |
|------|-------|-------------|
| 2009 | Rendle, S. et al. "BPR: Bayesian Personalized Ranking from Implicit Feedback." *UAI*. | BPR loss formulation |
| 2011 | Weston, J., Bengio, S. & Usunier, N. "WSABIE: Scaling Up to Large Vocabulary Image Annotation." *IJCAI*. | WARP loss |
| 2012 | Rendle, S. "Factorization Machines." *ICDM*. | Generalizes MF to arbitrary feature interactions |
| 2016 | He, X. et al. "Fast Matrix Factorization for Online Recommendation with Implicit Feedback." *SIGIR*. | Efficient ALS for implicit data (eALS) |

### 12.4 Deep Learning & Neural CF

| Year | Paper | Contribution |
|------|-------|-------------|
| 2016 | Covington, P. et al. "Deep Neural Networks for YouTube Recommendations." *RecSys*. | YouTube's two-stage DNN architecture |
| 2016 | Cheng, H. et al. "Wide & Deep Learning for Recommender Systems." *DLRS*. | Google Play's hybrid architecture |
| 2017 | He, X. et al. "Neural Collaborative Filtering." *WWW*. | NeuMF: GMF + MLP |
| 2018 | Liang, D. et al. "Variational Autoencoders for Collaborative Filtering." *WWW*. | VAE-CF (MultVAE) |
| 2019 | Sun, F. et al. "BERT4Rec: Sequential Recommendation with Bidirectional Encoder Representations from Transformers." *CIKM*. | Transformer-based sequential CF |

### 12.5 Graph-Based Collaborative Filtering

| Year | Paper | Contribution |
|------|-------|-------------|
| 2018 | Ying, R. et al. "Graph Convolutional Neural Networks for Web-Scale Recommender Systems." *KDD*. | PinSage (Pinterest) |
| 2019 | Wang, X. et al. "Neural Graph Collaborative Filtering." *SIGIR*. | NGCF |
| 2020 | He, X. et al. "LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation." *SIGIR*. | LightGCN (state-of-the-art graph CF) |
| 2021 | Mao, K. et al. "UltraGCN: Ultra Simplification of Graph Convolutional Networks for Recommendation." *CIKM*. | Infinite-layer approximation |
| 2022 | El-Kishky, A. et al. "TwHIN: Embedding the Twitter Heterogeneous Information Network for Personalized Recommendation." *KDD*. | Twitter's graph embedding |

### 12.6 Cold Start, Hybrid & Side Information

| Year | Paper | Contribution |
|------|-------|-------------|
| 2011 | Agarwal, D. & Chen, B. "Regression-based Latent Factor Models." *KDD*. | Feature-augmented MF for cold start |
| 2015 | Kula, M. "Metadata Embeddings for User and Item Cold-Start Recommendations." *LBR@RecSys*. | LightFM hybrid (BPR + WARP with features) |
| 2017 | Barkan, O. & Koenigstein, N. "Item2Vec: Neural Item Embedding for Collaborative Filtering." *MLSP*. | Word2Vec analogy for items |
| 2018 | Volkovs, M. et al. "DropoutNet: Addressing Cold Start in Recommender Systems." *NeurIPS*. | Dropout-based robustness to missing history |

### 12.7 Surveys & Textbooks

- **Ricci, F., Rokach, L. & Shapira, B.** (2022). *Recommender Systems Handbook* (3rd ed.). Springer. — The definitive reference textbook.
- **Aggarwal, C.** (2016). *Recommender Systems: The Textbook*. Springer. — Comprehensive mathematical treatment.
- **Zhang, S. et al.** (2019). "Deep Learning Based Recommender System: A Survey and New Perspectives." *ACM Computing Surveys* 52(1). — Survey of deep CF methods.
- **Wu, S. et al.** (2022). "Graph Neural Networks in Recommender Systems: A Survey." *ACM Computing Surveys* 55(5). — GNN-based CF comprehensive review.
- **Wang, S. et al.** (2021). "A Survey on Knowledge Graph-Based Recommender Systems." *IEEE TKDE* 34(8). — Knowledge-enhanced CF.

### 12.8 Open-Source Libraries & Datasets

**Libraries:**

| Library | Language | Algorithms | Use Case |
|---------|----------|-----------|----------|
| [Surprise](https://surpriselib.com/) | Python | SVD, KNN, NMF, Baseline | Research, prototyping |
| [LightFM](https://github.com/lyst/lightfm) | Python/Cython | BPR, WARP, hybrid | Cold start, side features |
| [Implicit](https://github.com/benfred/implicit) | Python/CUDA | ALS, BPR, LMF | Implicit feedback at scale |
| [RecBole](https://recbole.io/) | PyTorch | 90+ models (NCF, LightGCN, BERT4Rec) | Benchmarking, reproduction |
| [Spark MLlib ALS](https://spark.apache.org/docs/latest/ml-collaborative-filtering.html) | Scala/PySpark | ALS (explicit + implicit) | Distributed production |
| [Cornac](https://cornac.preferred.ai/) | Python | BPR, VAECF, BiVAECF, CTR | Multimodal recommendations |
| [Microsoft Recommenders](https://github.com/microsoft/recommenders) | Python | SAR, ALS, NCF, xDeepFM, LightGCN | End-to-end pipelines |

**Benchmark Datasets:**

| Dataset | Domain | Scale | Type |
|---------|--------|-------|------|
| [MovieLens 25M](https://grouplens.org/datasets/movielens/) | Movies | 25M ratings, 162K users | Explicit |
| [Amazon Reviews (2023)](https://amazon-reviews-2023.github.io/) | E-commerce | 571M reviews, 54 categories | Explicit + implicit |
| [Yelp Open Dataset](https://www.yelp.com/dataset) | Local business | 7M reviews | Explicit |
| [Gowalla / Foursquare](https://snap.stanford.edu/data/) | Location | 6.4M check-ins | Implicit |
| [Last.fm 1K](http://www.dtic.upf.edu/~ocelma/MusicRecommendationDataset/) | Music | 19M play counts | Implicit |
| [Pinterest](https://github.com/hexiangnan/neural_collaborative_filtering) | Images | 1.5M interactions | Implicit (binary) |
| [KuaiRec](https://kuairec.com/) | Short video | 12M interactions, full matrix | Implicit + dense |

### 12.9 Industry Engineering Blogs

- **Netflix**: ["System Architectures for Personalization and Recommendation"](https://netflixtechblog.com/) — Production ML systems at Netflix scale.
- **Spotify**: ["Music Recommendations at Scale"](https://engineering.atspotify.com/) — Implicit ALS, exploration, and audio features.
- **Pinterest**: ["PinSage: Graph Neural Networks for Web-Scale Recommender Systems"](https://medium.com/pinterest-engineering) — GNN at 3B node scale.
- **YouTube**: ["Deep Neural Networks for YouTube Recommendations"](https://research.google/) — Two-tower architecture.
- **Airbnb**: ["Listing Embeddings in Search Ranking"](https://medium.com/airbnb-engineering) — Session-based embedding learning.
- **Twitter/X**: ["TwHIN: Twitter's Open-Source Recommendation Algorithm"](https://blog.twitter.com/engineering) — Heterogeneous graph embeddings.
- **Uber Eats**: ["Food Discovery with Uber Eats"](https://eng.uber.com/) — Graph learning for food recommendations.

---

*This notebook was prepared as a comprehensive reference for collaborative filtering algorithms — from classical neighborhood methods to state-of-the-art graph neural networks. Each section includes mathematical foundations, working implementations, and real-world industrial deployments.*